<a href="https://colab.research.google.com/github/deepakra123/Multimodal-Deep-Learning-Approach-for-Emotion-Aware-Speech-Recognition/blob/main/Multimodal_Emotion_Aware_Speech_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Installations
# We install specific libraries for efficient fine-tuning of large models (PEFT)
!pip install --upgrade pip
!pip install --upgrade transformers datasets accelerate peft bitsandbytes
!pip install jiwer

print(" All libraries installed successfully!")

In [ ]:
 # Cell 2: Hugging Face Login
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# Cell 3: Load a Small Data Sample
from datasets import load_dataset, Audio

print("--- Loading a small 1% sample of the LibriSpeech dataset ---")

# Load just a tiny fraction for a quick test run
dataset = load_dataset("librispeech_asr", "clean", split="train.100[:1%]")

# Split the small sample into a training and test set
dataset = dataset.train_test_split(test_size=0.2)

# Resample the audio to 16kHz, as required by Whisper
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

print("\n Small dataset sample loaded successfully!")
print(dataset)

In [ ]:
!pip install torchcodec
!pip install soundfile
!pip install audioread


In [ ]:

# Cell 4: Load Whisper Processor and Prepare the Dataset
from transformers import WhisperProcessor

print("--- Loading Whisper Processor ---")
processor = WhisperProcessor.from_pretrained("openai/whisper-base.en", language="English", task="transcribe")

def prepare_dataset(batch):
    # This function is much simpler thanks to the powerful WhisperProcessor.
    audio = batch["audio"]
    # The processor elegantly handles feature extraction and tokenization.
    batch["input_features"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor(text=batch["text"].upper()).input_ids
    return batch

print("\n--- Preprocessing the dataset sample ---")
# This will apply the preparation function to our small dataset.
processed_dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names["train"], num_proc=2)

print("\n Dataset sample preprocessed successfully!")

In [ ]:
import importlib
spec = importlib.util.find_spec("torchcodec")
print("Torchcodec found:", spec is not None)


In [ ]:
# Cell 5: Load and Configure Whisper Model for LoRA Fine-Tuning
from transformers import WhisperForConditionalGeneration
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

print("--- Loading Whisper Model and configuring for LoRA ---")

# 1. Load the pre-trained model
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base.en", load_in_8bit=True)

# 2. Prepare the model for 8-bit training
model = prepare_model_for_kbit_training(model)

# 3. Define the LoRA configuration
# This tells the system which parts of the model to adapt.
config = LoraConfig(r=8, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")

# 4. Wrap the model with the PEFT configuration
model = get_peft_model(model, config)
model.print_trainable_parameters()

print("\n Whisper model is configured for efficient fine-tuning!")

In [ ]:
# Cell 6: Configure and Start the Training (Backwards-Compatible)
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# The Data Collator correctly prepares batches for the Whisper model.
class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# --- THE FIX IS HERE ---
# We are replacing the modern 'evaluation_strategy' argument
# with the older 'do_eval=True' argument.
# We also explicitly set the 'save_strategy' to match the evaluation frequency.
print("--- Defining Training Arguments with backwards-compatible syntax ---")
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper_finetuned",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    num_train_epochs=5,

    # OLD, COMPATIBLE SYNTAX
    do_eval=True,
    save_strategy="epoch", # Save a checkpoint at the end of each epoch

    per_device_eval_batch_size=8,
)

# Initialize the Trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["test"],
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
)

print("\n--- Starting the fine-tuning process ---")
trainer.train()

print("\n Training on the small sample is complete!")

# Save the trained LoRA adapter.
final_model_path = "./whisper_model_final_adapter"
trainer.save_model(final_model_path)
print(f"\n Final fine-tuned model adapter saved to: {final_model_path}")

In [ ]:
# --- Cell: Evaluate the Fine-Tuned Whisper Model ---
from datasets import load_metric
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
from tqdm import tqdm

# 1. Load evaluation metrics
import evaluate
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")
bleu_metric = evaluate.load("bleu")

print("--- Evaluating Whisper Model ---")

# 2. Generate predictions on the test dataset
predictions = []
references = []

for example in tqdm(processed_dataset["test"]):
    input_features = torch.tensor(example["input_features"]).unsqueeze(0).to(model.device)
    predicted_ids = model.generate(input_features)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    reference = processor.tokenizer.decode(example["labels"], skip_special_tokens=True)
    predictions.append(transcription.lower())
    references.append(reference.lower())

# 3. Compute Metrics
wer = wer_metric.compute(predictions=predictions, references=references)
cer = cer_metric.compute(predictions=predictions, references=references)
bleu = bleu_metric.compute(predictions=predictions, references=references)

print(f"\nWord Error Rate (WER): {wer:.4f}")
print(f"Character Error Rate (CER): {cer:.4f}")
print(f"BLEU Score: {bleu['bleu']:.4f}")

# --- Optional: If you have fixed vocabulary, you can compute confusion matrix ---
# Convert to word-level comparison
y_true = []
y_pred = []

for ref, pred in zip(references, predictions):
    ref_words = ref.split()
    pred_words = pred.split()
    for r, p in zip(ref_words, pred_words[:len(ref_words)]):
        y_true.append(r)
        y_pred.append(p)

if len(y_true) > 0 and len(set(y_true)) < 100:  # Avoid huge vocab confusion matrices
    cm = confusion_matrix(y_true, y_pred, labels=list(set(y_true)))
    cm_df = pd.DataFrame(cm, index=list(set(y_true)), columns=list(set(y_true)))
    plt.figure(figsize=(10,8))
    sns.heatmap(cm_df, annot=False, cmap='Blues')
    plt.title("Word-Level Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    print("\nClassification Report (word-level):")
    print(classification_report(y_true, y_pred))

# --- Correlation heatmap (optional) ---
# Example: Correlation between audio feature means and error rates
feature_errors = []
feature_means = []

for i, ex in enumerate(processed_dataset["test"]):
    feat_mean = np.mean(ex["input_features"])
    err = np.mean([1 if a != b else 0 for a, b in zip(references[i].split(), predictions[i].split())])
    feature_means.append(feat_mean)
    feature_errors.append(err)

if len(feature_means) > 1:
    corr_df = pd.DataFrame({"FeatureMean": feature_means, "ErrorRate": feature_errors})
    plt.figure(figsize=(6,4))
    sns.heatmap(corr_df.corr(), annot=True, cmap="coolwarm")
    plt.title("Correlation Heatmap: Feature Mean vs Error Rate")
    plt.show()

print("\n--- Evaluation Complete ---")


In [ ]:
#Task-Level ROC Curve
from sklearn.metrics import roc_curve, auc, precision_score, recall_score, f1_score, accuracy_score
import matplotlib.pyplot as plt
import numpy as np

# Example: Binary correctness (1 if predicted exactly matches reference, else 0)
y_true_binary = []
y_scores = []  # Confidence proxy (we can use normalized edit distance or log-likelihoods if available)

for ref, pred in zip(references, predictions):
    correct = 1 if ref.strip() == pred.strip() else 0
    y_true_binary.append(correct)
    # Approximate confidence: inverse of CER per sentence
    cer_sent = cer_metric.compute(predictions=[pred], references=[ref])
    conf_score = 1 - cer_sent  # higher = better
    y_scores.append(conf_score)

# Convert to numpy arrays
y_true_binary = np.array(y_true_binary)
y_scores = np.array(y_scores)

# Compute ROC and AUC
fpr, tpr, thresholds = roc_curve(y_true_binary, y_scores)
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Sentence-Level ASR Accuracy)')
plt.legend(loc="lower right")
plt.show()

# Additional metrics
precision = precision_score(y_true_binary, y_scores > 0.5)
recall = recall_score(y_true_binary, y_scores > 0.5)
f1 = f1_score(y_true_binary, y_scores > 0.5)
accuracy = accuracy_score(y_true_binary, y_scores > 0.5)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {roc_auc:.4f}")


In [ ]:
#Word Level ROC
from sklearn.metrics import roc_curve, auc

word_true = []
word_scores = []

for ref, pred in zip(references, predictions):
    ref_words = ref.split()
    pred_words = pred.split()
    for i, word in enumerate(ref_words):
        if i < len(pred_words):
            correct = 1 if word == pred_words[i] else 0
            word_true.append(correct)
            # Approximate confidence (similar to before)
            word_scores.append(1 - cer_metric.compute(predictions=[pred_words[i]], references=[word]))
        else:
            word_true.append(0)
            word_scores.append(0)

# Compute ROC
fpr, tpr, _ = roc_curve(word_true, word_scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color='green', lw=2, label=f'Word-Level ROC (AUC = {roc_auc:.2f})')
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Word-Level ROC Curve')
plt.legend()
plt.show()


In [ ]:
# Cell: Mount Google Drive
from google.colab import drive
import os

print("--- Mounting Google Drive ---")
drive.mount('/content/drive')

# This is the folder in your Drive where everything will be saved.
# We will use this 'gdrive_path' variable in the next step.
gdrive_path = "/content/drive/MyDrive/ASR_Project_Whisper"

# Create the folder if it doesn't exist
os.makedirs(gdrive_path, exist_ok=True)
print(f"\n Google Drive mounted. All project files will be saved in: {gdrive_path}")

In [ ]:
# Cell 6: Configure and Start Training (BACKWARDS-COMPATIBLE & Saving to Google Drive)
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import os

# --- The Data Collator (This part is correct and remains the same) ---
class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor
    def __call__(self, features):
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


# --- THE FIX IS IN THIS SECTION ---
output_dir_path = os.path.join(gdrive_path, "whisper_finetuned_checkpoints")
print(f"--- Defining Training Arguments with backwards-compatible syntax ---")

# We calculate the number of steps per epoch to mimic the "epoch" strategy
steps_per_epoch = len(processed_dataset["train"]) // (8 * 2) # train_batch_size * gradient_accumulation

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir_path,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    num_train_epochs=5,
    per_device_eval_batch_size=8,

    # --- REPLACED MODERN ARGUMENTS WITH OLD, COMPATIBLE SYNTAX ---
    # Instead of evaluation_strategy="epoch", we use:
    do_eval=True,
    eval_steps=steps_per_epoch, # Evaluate at the end of each epoch

    # Instead of save_strategy="epoch", we use:
    save_steps=steps_per_epoch, # Save a checkpoint at the end of each epoch
)

# Initialize the Trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["test"],
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
)

print("\n--- Starting the fine-tuning process ---")
trainer.train()

print("\n Training is complete!")

# --- This final save part is correct and remains the same ---
final_model_path = os.path.join(gdrive_path, "whisper_model_final_adapter")
trainer.save_model(final_model_path)
print(f"\n Final fine-tuned model adapter saved successfully to your Google Drive at: {final_model_path}")

Testing


In [ ]:
pip install -U bitsandbytes

In [ ]:
# Cell: Load the Fine-Tuned Model for Inference (MODERN & CORRECTED)
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
import os

print("--- Loading the fine-tuned Whisper model with modern syntax ---")

# Define the path to your saved adapter in Google Drive
final_model_path = os.path.join(gdrive_path, "whisper_model_final_adapter")
base_model_name = "openai/whisper-base.en"

# --- FIX #1: Use the modern quantization config ---
# Instead of `load_in_8bit=True`, we create a config object.
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

# 1. Load the base Whisper model with the new quantization config
base_model = WhisperForConditionalGeneration.from_pretrained(
    base_model_name,
    quantization_config=quantization_config
)

# 2. Load the Whisper processor (this remains the same)
processor = WhisperProcessor.from_pretrained(base_model_name)

# --- FIX #2: Use the modern PeftModel.from_pretrained method ---
# Instead of `get_peft_model`, we now load the adapter directly onto the base model.
print("\nMerging base model with the fine-tuned LoRA adapter...")
inference_model = PeftModel.from_pretrained(base_model, final_model_path)

# 3. Set the device (GPU) and put the model in evaluation mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model.to(device)
inference_model.eval()

print(f"\n Fine-tuned model loaded successfully and is ready for inference on {device}!")

In [ ]:
# Cell: Upload a test audio file
from google.colab import files
import torchaudio

print("Please upload a short WAV or MP3 audio file to test.")
uploaded = files.upload()

# Get the name of the uploaded file
test_audio_path = list(uploaded.keys())[0]

print(f"\nFile '{test_audio_path}' uploaded successfully.")

In [ ]:
# Cell: Perform Inference and Get Transcription (CORRECTED)
import torchaudio
import torch # Make sure torch is imported

print("--- Transcribing the audio file... ---")

# 1. Load the audio file into a tensor
waveform, sample_rate = torchaudio.load(test_audio_path)

# 2. Resample to 16kHz if necessary
if sample_rate != 16000:
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
    waveform = resampler(waveform)

# 3. Process the audio to get the input features
input_features = processor(
    waveform.squeeze().numpy(),
    sampling_rate=16000,
    return_tensors="pt"
).input_features.to(device)


# --- THIS IS THE ONE-LINE FIX ---
# We convert our input data to half-precision (float16) to match the model's internal data type.
input_features = input_features.to(torch.float16)


# 4. Generate the transcription
with torch.no_grad():
    predicted_ids = inference_model.generate(input_features)

# 5. Decode the predicted token IDs back to a text string
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

# --- Display the results ---
print("\n" + "="*50)
print("              RESULTS")
print("="*50)
if 'ground_truth_text' in locals():
    print(f"\nORIGINAL TEXT:\n'{ground_truth_text}'")

print(f"\nMODEL'S PREDICTION:\n'{transcription.upper()}'")
print("="*50)

In [ ]:
pip install -U bitsandbytes

In [ ]:
# Cell 1: Setup Environment
print("--- Installing required libraries ---")
!pip install --upgrade transformers datasets accelerate peft bitsandbytes
!pip install torch torchaudio --quiet

from google.colab import drive
from huggingface_hub import notebook_login
import os

print("\n--- Authenticating with Hugging Face ---")
notebook_login()

print("\n--- Mounting Google Drive ---")
drive.mount('/content/drive')

# IMPORTANT: This path must match the folder where you saved your project
gdrive_path = "/content/drive/MyDrive/ASR_Project_Whisper"
print(f"\n Setup complete. Project path is set to: {gdrive_path}")

In [ ]:
# Cell 2: Load the Fine-Tuned Model for Inference
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
import os

print("--- Loading the fine-tuned Whisper model from Google Drive ---")

# Define the path to your saved adapter in Google Drive
final_model_path = os.path.join(gdrive_path, "whisper_model_final_adapter")
base_model_name = "openai/whisper-base.en"

# Use the modern BitsAndBytesConfig for 8-bit loading
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

# 1. Load the base Whisper model with the quantization config
print("Loading base model...")
base_model = WhisperForConditionalGeneration.from_pretrained(
    base_model_name,
    quantization_config=quantization_config
)

# 2. Load the Whisper processor
print("Loading processor...")
processor = WhisperProcessor.from_pretrained(base_model_name)

# 3. Merge the base model with your fine-tuned LoRA adapter
print("Merging base model with LoRA adapter...")
inference_model = PeftModel.from_pretrained(base_model, final_model_path)

# 4. Set the device (GPU) and put the model in evaluation mode
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model.to(device)
inference_model.eval()

print(f"\n Fine-tuned model loaded successfully and is ready for inference on {device}!")

In [ ]:
# Cell 3, Option A: Upload a test audio file
from google.colab import files

print("Please upload a short WAV or MP3 audio file to test.")
uploaded = files.upload()

# Get the name of the uploaded file
test_audio_path = list(uploaded.keys())[0]

print(f"\nFile '{test_audio_path}' uploaded successfully.")

In [ ]:
# Cell: Perform Inference and Get Transcription (CORRECTED)
import torchaudio
import torch # Make sure torch is imported

print("--- Transcribing the audio file... ---")

# 1. Load the audio file into a tensor
waveform, sample_rate = torchaudio.load(test_audio_path)

# 2. Resample to 16kHz if necessary
if sample_rate != 16000:
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
    waveform = resampler(waveform)

# 3. Process the audio to get the input features
input_features = processor(
    waveform.squeeze().numpy(),
    sampling_rate=16000,
    return_tensors="pt"
).input_features.to(device)


# --- THIS IS THE ONE-LINE FIX ---
# We convert our input data to half-precision (float16) to match the model's internal data type.
input_features = input_features.to(torch.float16)


# 4. Generate the transcription
with torch.no_grad():
    predicted_ids = inference_model.generate(input_features)

# 5. Decode the predicted token IDs back to a text string
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

# --- Display the results ---
print("\n" + "="*50)
print("              RESULTS")
print("="*50)
if 'ground_truth_text' in locals():
    print(f"\nORIGINAL TEXT:\n'{ground_truth_text}'")

print(f"\nMODEL'S PREDICTION:\n'{transcription.upper()}'")
print("="*50)

##Feature 2 Trial


In [ ]:
# Cell 1: Setup Environment
print("--- Installing required libraries ---")
!pip install torch datasets numpy tqdm requests scikit-learn --quiet

from google.colab import drive
import os

print("\n--- Mounting Google Drive ---")
drive.mount('/content/drive')

# This is the folder in your Drive where everything will be saved.
gdrive_path = "/content/drive/MyDrive/DL_Course_Project"
os.makedirs(gdrive_path, exist_ok=True)
print(f"\n Setup complete. Project folder is set to: {gdrive_path}")

In [ ]:
# Cell 2: Download and Load GloVe Embeddings
import requests
import zipfile
import io
import numpy as np

print("--- Step 2: Downloading and Loading GloVe Word Embeddings ---")

# Download the GloVe vectors (6B tokens, 100 dimensions)
glove_zip_url = "http://nlp.stanford.edu/data/glove.6B.zip"
r = requests.get(glove_zip_url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall("/content/glove")
print("GloVe data downloaded and extracted.")

# Load the 100-dimensional vectors into a dictionary
glove_file_path = "/content/glove/glove.6B.100d.txt"
glove_embeddings = {}
with open(glove_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        glove_embeddings[word] = vector

print(f"\n Loaded {len(glove_embeddings)} word vectors into memory.")

In [ ]:
# Cell 3: Create Sentence Embeddings using GloVe Averaging
from datasets import load_dataset
from tqdm.auto import tqdm
import torch

print("--- Step 3: Creating sentence embeddings by averaging GloVe vectors ---")

# 1. Load the dialogue dataset, caching to your Google Drive
cache_path = os.path.join(gdrive_path, "dataset_cache")
dataset = load_dataset("knkarthick/dialogsum", split="train", cache_dir=cache_path)

# 2. Extract individual sentences
sentences = []
for item in dataset:
    sentences.extend(item['dialogue'].split('\n'))
# Convert to lowercase to match GloVe's vocabulary
sentences = [s.strip().lower() for s in sentences if s.strip()]

# 3. Create sentence embeddings
sentence_embeddings_list = []
embedding_dim = 100 # This must match the GloVe file we loaded (100d)
# Create a zero vector for any words not found in the GloVe dictionary
zero_vector = np.zeros(embedding_dim)

for sentence in tqdm(sentences, desc="Processing sentences"):
    words = sentence.split()
    # Get the vector for each word, using the zero vector as a default
    word_vectors = [glove_embeddings.get(word, zero_vector) for word in words]

    if len(word_vectors) > 0:
        # Calculate the mean of the word vectors to get the sentence vector
        sentence_vector = np.mean(word_vectors, axis=0)
        sentence_embeddings_list.append(sentence_vector)

# Convert our list of vectors into a single PyTorch tensor for the model
sentence_embeddings = torch.FloatTensor(np.array(sentence_embeddings_list))

print("\n All sentences have been converted into GloVe-based vectors!")
print(f"Shape of final sentence embeddings tensor: {sentence_embeddings.shape}")

In [ ]:
# Cell 4: Define and Train the Autoencoder
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# --- Define the Model Architecture ---
class SentenceAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # The Encoder compresses the 100-dim input to a 64-dim "meaning vector"
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )
        # The Decoder tries to reconstruct the original 100-dim vector
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

print("--- Step 4: Starting the Autoencoder training loop ---")

# --- Prepare for Training ---
dataset_torch = TensorDataset(sentence_embeddings)
dataloader = DataLoader(dataset_torch, batch_size=64, shuffle=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceAutoencoder(input_dim=embedding_dim)
model.to(device)
criterion = nn.MSELoss() # Mean Squared Error is the standard loss for reconstruction
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# --- The Training Loop ---
num_epochs = 20 # Train for 20 epochs for good results
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc=f"Epoch {epoch + 1}/{num_epochs}", leave=False):
        vectors = batch[0].to(device)
        reconstructed = model(vectors)
        loss = criterion(reconstructed, vectors)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"🎉 Epoch {epoch + 1} Complete! Average Reconstruction Loss: {avg_loss:.6f}")

print("\n Training complete!")

In [ ]:
# Cell 5: Save the Trained Encoder
print("--- Step 5: Saving the final trained encoder to Google Drive ---")

# Define the final save path using the 'gdrive_path' variable
encoder_save_path = os.path.join(gdrive_path, "summarizer_encoder_glove.pt")

# Save only the state dictionary of the encoder part of our model
torch.save(model.encoder.state_dict(), encoder_save_path)

print(f"\n GloVe-based encoder model saved successfully to: '{encoder_save_path}'")

##Testing

In [ ]:
# Cell 1: Setup Environment
print("--- Installing required libraries ---")
!pip install torch numpy tqdm requests scikit-learn --quiet

from google.colab import drive
import os

print("\n--- Mounting Google Drive ---")
drive.mount('/content/drive')

# IMPORTANT: This path must match the folder where you saved your project
gdrive_path = "/content/drive/MyDrive/DL_Course_Project"
print(f"\n Setup complete. Project path is set to: {gdrive_path}")

In [ ]:
# Cell 2: Load All Inference Tools
import requests
import zipfile
import io
import numpy as np
import torch
import torch.nn as nn

print("--- Step 2: Loading GloVe embeddings and your trained encoder model ---")

# --- Part A: Load GloVe Embeddings ---
glove_zip_url = "http://nlp.stanford.edu/data/glove.6B.zip"
r = requests.get(glove_zip_url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall("/content/glove")
glove_file_path = "/content/glove/glove.6B.100d.txt"
glove_embeddings = {}
with open(glove_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        glove_embeddings[word] = vector
print(f" Loaded {len(glove_embeddings)} GloVe word vectors.")

# --- Part B: Load Your Trained Encoder ---
# 1. We must first define the model architecture EXACTLY as we did during training.
embedding_dim = 100 # Must match the GloVe vectors
class SentenceAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# 2. Instantiate the encoder part of the model
encoder_model = SentenceAutoencoder(input_dim=embedding_dim).encoder

# 3. Load the saved weights from your Google Drive
encoder_path = os.path.join(gdrive_path, "summarizer_encoder_glove.pt")
encoder_model.load_state_dict(torch.load(encoder_path))

# 4. Set the model to evaluation mode and move to device
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder_model.to(device)
encoder_model.eval()
print(f"\n Trained encoder loaded successfully and is ready for inference on {device}!")

In [ ]:
# Cell 3: Define Input Text
print("--- Step 3: Defining a sample conversation to summarize ---")

conversation_text = """
Alex: Okay team, let's sync up on the Q4 marketing launch. Where are we with the ad creatives?
Brenda: The video ad is in final review. We should have the final cut by tomorrow afternoon.
Carl: I've drafted the copy for the social media posts. They are in the shared drive for feedback.
Alex: Great. Brenda, what about the landing page?
Brenda: The design is approved. The core challenge is the slow loading time on mobile. We need engineering support.
Alex: Okay, that's a critical blocker. I'll flag this to the engineering lead immediately. Carl, anything else?
Carl: No, I'm just waiting for the final video link to embed in the posts.
Alex: Perfect. So the main action item is resolving the landing page performance issue. Let's regroup on that tomorrow.
"""

print("Sample conversation is ready.")

In [ ]:
# Cell 4: Perform Inference to Find the Extractive Summary
print("--- Step 4: Finding the most representative sentence ---")

# 1. Process the input text just like we did during training
sentences = [s.strip().lower() for s in conversation_text.strip().split('\n') if s.strip()]
zero_vector = np.zeros(embedding_dim)
sentence_vectors_list = []
for s in sentences:
    words = s.split()
    word_vectors = [glove_embeddings.get(word, zero_vector) for word in words]
    if len(word_vectors) > 0:
        sentence_vectors_list.append(np.mean(word_vectors, axis=0))

# Convert to a PyTorch tensor
sentence_vectors = torch.FloatTensor(np.array(sentence_vectors_list)).to(device)

# 2. Use your trained encoder to get the "meaning vectors"
with torch.no_grad():
    encoded_vectors = encoder_model(sentence_vectors)

# 3. Calculate the "average meaning" of the conversation (the centroid)
centroid = torch.mean(encoded_vectors, dim=0)

# 4. Find the sentence whose meaning vector is closest to the centroid
# We calculate the distance of each sentence from the centroid
distances = torch.linalg.norm(encoded_vectors - centroid, dim=1)
# Find the index of the sentence with the smallest distance
summary_index = torch.argmin(distances).item()

# 5. Get the summary sentence!
summary_sentence = sentences[summary_index].split(': ', 1)[-1] # Clean up the speaker name

# --- Display the results in a clear, explainable way ---
print("\n" + "="*50)
print("              RESULTS")
print("="*50)
print("\n--- Full Conversation ---")
# Print the original sentences for context
for i, sentence in enumerate(sentences):
    # Highlight the chosen summary sentence
    if i == summary_index:
        print(f"\n>>> ** {sentence} ** <<<\n") # Using markdown for emphasis
    else:
        print(sentence)

print("\n" + "="*50)
print(f"\nEXPLAINABLE EXTRACTIVE SUMMARY:\n'{summary_sentence.capitalize()}'")
print("="*50)

#Possible Feature 2

In [ ]:
# Cell 1: Setup Environment
print("--- Installing required libraries ---")
!pip install torch datasets numpy tqdm scikit-learn --quiet

from google.colab import drive
import os

print("\n--- Mounting Google Drive ---")
drive.mount('/content/drive')

gdrive_path = "/content/drive/MyDrive/DL_Course_Project"
os.makedirs(gdrive_path, exist_ok=True)
print(f"\n Setup complete. Project folder is set to: {gdrive_path}")

In [ ]:
# Re-run this cell!
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# Cell: Create the Sentence Importance Dataset
from datasets import load_dataset
from tqdm.auto import tqdm
import torch
import numpy as np
import os

print("--- Step 2: Creating a Labeled Dataset for the Sentence Scoring RNN ---")

# --- Part A: Load Tools (GloVe and DialogSum) ---
# This re-uses the GloVe loading code from our previous implementation.
if 'glove_embeddings' not in locals():
    print("Loading GloVe embeddings...")
    glove_zip_url = "http://nlp.stanford.edu/data/glove.6B.zip"
    import requests, zipfile, io
    r = requests.get(glove_zip_url)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall("/content/glove")
    glove_file_path = "/content/glove/glove.6B.100d.txt"
    glove_embeddings = {}
    with open(glove_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_embeddings[word] = vector
    print(f"Loaded {len(glove_embeddings)} GloVe vectors.")

cache_path = os.path.join(gdrive_path, "dataset_cache")
dataset = load_dataset("knkarthick/dialogsum", split="train[:30%]", cache_dir=cache_path)

# --- Part B: Generate Labels using Cosine Similarity ---
embedding_dim = 100
zero_vector = np.zeros(embedding_dim)

def sentence_to_vector(sentence):
    """Helper function to convert a sentence to a GloVe-averaged vector."""
    words = sentence.lower().split()
    word_vectors = [glove_embeddings.get(word, zero_vector) for word in words]
    return np.mean(word_vectors, axis=0) if len(word_vectors) > 0 else zero_vector

X_rnn = [] # Input vectors for the RNN
y_rnn = [] # Labels (1 or 0) for the RNN

# A threshold for similarity. If a sentence is this close to the summary, it's important.
SIMILARITY_THRESHOLD = 0.7

for example in tqdm(dataset, desc="Generating sentence-level labels"):
    summary_vector = sentence_to_vector(example['summary'])
    dialogue_sentences = [s.strip() for s in example['dialogue'].split('\n') if s.strip()]

    for sentence in dialogue_sentences:
        sentence_vector = sentence_to_vector(sentence)

        # Calculate cosine similarity between the sentence and the summary
        cos_sim = np.dot(summary_vector, sentence_vector) / (np.linalg.norm(summary_vector) * np.linalg.norm(sentence_vector))

        # If similarity is high, label it as important (1)
        if cos_sim > SIMILARITY_THRESHOLD:
            y_rnn.append(1)
        else:
            y_rnn.append(0)

        X_rnn.append(sentence_vector)

# Convert to PyTorch Tensors
X_rnn_tensor = torch.FloatTensor(np.array(X_rnn))
y_rnn_tensor = torch.FloatTensor(np.array(y_rnn))

print(f"\n Created dataset for RNN with {len(X_rnn)} sentences.")
print(f"Number of 'Important' sentences (Label 1): {sum(y_rnn)}")

In [ ]:
# Cell: Define and Train the Sentence Scoring RNN
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# --- Define the RNN Model ---
class SentenceScorerRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, n_layers=1):
        super().__init__()
        # We wrap the GloVe vector in a sequence of length 1 to use an RNN
        self.gru = nn.GRU(input_dim, hidden_dim, n_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1) # Output a single score

    def forward(self, x):
        # x shape: [batch_size, feature_dim]
        # We add a sequence dimension: [batch_size, 1, feature_dim]
        x = x.unsqueeze(1)
        gru_out, _ = self.gru(x)
        # We only care about the final output of the sequence
        output = self.fc(gru_out[:, -1, :])
        return output.squeeze(-1)


# --- Prepare for Training ---
dataset_rnn_torch = TensorDataset(X_rnn_tensor, y_rnn_tensor)
dataloader_rnn = DataLoader(dataset_rnn_torch, batch_size=64, shuffle=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

model_rnn = SentenceScorerRNN(input_dim=embedding_dim).to(device)
criterion_rnn = nn.BCEWithLogitsLoss()
optimizer_rnn = torch.optim.Adam(model_rnn.parameters(), lr=1e-3)

# --- The Training Loop ---
print("--- Starting RNN Training for Sentence Scoring ---")
num_epochs = 10
for epoch in range(num_epochs):
    model_rnn.train()
    total_loss = 0
    for inputs, labels in tqdm(dataloader_rnn, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_rnn.zero_grad()
        predictions = model_rnn(inputs)
        loss = criterion_rnn(predictions, labels)
        loss.backward()
        optimizer_rnn.step()
        total_loss += loss.item()
    print(f"🎉 Epoch {epoch+1} Complete! Average Loss: {total_loss/len(dataloader_rnn):.4f}")

# --- Save the Model ---
rnn_model_path = os.path.join(gdrive_path, "sentence_scorer_rnn.pt")
torch.save(model_rnn.state_dict(), rnn_model_path)
print(f"\n Trained Sentence Scoring RNN saved to: {rnn_model_path}")

In [ ]:
# --- Cell: Evaluate Sentence Scoring RNN ---
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import torch

print("\n--- Evaluating the Sentence Scoring RNN ---")

# 1️⃣  Prepare data for evaluation
model_rnn.eval()
with torch.no_grad():
    y_pred_logits = model_rnn(X_rnn_tensor.to(device)).cpu().numpy()
    y_pred_probs = torch.sigmoid(torch.tensor(y_pred_logits)).numpy()
    y_pred_labels = (y_pred_probs > 0.5).astype(int)
    y_true = y_rnn_tensor.numpy().astype(int)

# 2️⃣  Classification metrics
print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred_labels, target_names=["Not Important (0)", "Important (1)"]))

# 3️⃣  Confusion Matrix
cm = confusion_matrix(y_true, y_pred_labels)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Not Important", "Important"], yticklabels=["Not Important", "Important"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

# 4️⃣  ROC Curve & AUC
fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
auc_score = roc_auc_score(y_true, y_pred_probs)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc_score:.4f})', color='blue')
plt.plot([0,1], [0,1], 'r--', label='Chance Level')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Sentence Importance RNN')
plt.legend()
plt.grid(True)
plt.show()

# 5️⃣  Additional Summary Stats
acc = accuracy_score(y_true, y_pred_labels)
prec = precision_score(y_true, y_pred_labels)
rec = recall_score(y_true, y_pred_labels)
f1 = f1_score(y_true, y_pred_labels)

print("\n--- Summary of Key Metrics ---")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"AUC Score : {auc_score:.4f}")

# 6️⃣  Correlation Heatmap: between predictions and input feature vectors
# Compute mean of each feature and correlation with predicted probability
X_df = pd.DataFrame(X_rnn_tensor.numpy())
X_df["predicted_prob"] = y_pred_probs
corr_matrix = X_df.corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap: GloVe Feature Dimensions vs Predicted Importance")
plt.show()

# 7️⃣  Feature Importance (approx via permutation feature importance)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression

# Train a simple logistic model on same data to approximate interpretability
clf = LogisticRegression(max_iter=500)
clf.fit(X_rnn_tensor.numpy(), y_true)
result = permutation_importance(clf, X_rnn_tensor.numpy(), y_true, n_repeats=5, random_state=42)

importances = result.importances_mean
indices = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(8,4))
plt.bar(range(len(indices)), importances[indices])
plt.xticks(range(len(indices)), [f"Feature {i}" for i in indices])
plt.title("Top 10 Most Important GloVe Dimensions (Approx.)")
plt.show()

print("\n--- Evaluation Complete ---")


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
precision, recall, _ = precision_recall_curve(y_true, y_pred_probs)
ap = average_precision_score(y_true, y_pred_probs)
plt.plot(recall, precision, color='purple', label=f'PR curve (AP = {ap:.4f})')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision–Recall Curve')
plt.legend(); plt.show()


In [ ]:
#To Check if predicted probabilities are reliable
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(y_true, y_pred_probs, n_bins=10)
plt.plot(prob_pred, prob_true, marker='o')
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel('Predicted Probability')
plt.ylabel('Observed Frequency')
plt.title('Calibration Curve')
plt.show()


In [ ]:
#Distribution Plot of Predicted Scores
sns.histplot(y_pred_probs, bins=30, kde=True)
plt.title("Distribution of Predicted Sentence Importance Scores")
plt.show()


In [ ]:
# Cell: Define and Train the CNN for Sentence Importance
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print("\n--- Training CNN for Sentence Importance Scoring ---")

# --- Step 1: Prepare Dataset ---
# CNN expects [batch_size, channels, sequence_length]
# Since each sentence is represented as a single averaged vector (100-d),
# we reshape it to [batch, 1, embedding_dim]
X_cnn_tensor = X_rnn_tensor.unsqueeze(1)  # Add channel dimension
dataset_cnn = TensorDataset(X_cnn_tensor, y_rnn_tensor)
dataloader_cnn = DataLoader(dataset_cnn, batch_size=64, shuffle=True)

# --- Step 2: Define CNN Model ---
class SentenceScorerCNN(nn.Module):
    def __init__(self, embedding_dim=100, num_filters=128, kernel_sizes=[2,3,4]):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=k)
            for k in kernel_sizes
        ])
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        # x shape: [batch, 1, embedding_dim]
        conv_outs = []
        for conv in self.convs:
            c = torch.relu(conv(x))  # [batch, num_filters, L_out]
            c = torch.max(c, dim=2)[0]  # Global max pooling → [batch, num_filters]
            conv_outs.append(c)
        out = torch.cat(conv_outs, dim=1)
        out = self.dropout(out)
        return self.fc(out).squeeze(-1)

# --- Step 3: Initialize and Train ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model_cnn = SentenceScorerCNN(embedding_dim=100).to(device)
criterion_cnn = nn.BCEWithLogitsLoss()
optimizer_cnn = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)

# --- Step 4: Train ---
num_epochs = 10
for epoch in range(num_epochs):
    model_cnn.train()
    total_loss = 0
    for inputs, labels in dataloader_cnn:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_cnn.zero_grad()
        predictions = model_cnn(inputs)
        loss = criterion_cnn(predictions, labels)
        loss.backward()
        optimizer_cnn.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{num_epochs} - CNN Loss: {total_loss/len(dataloader_cnn):.4f}")

# --- Step 5: Save Model ---
cnn_model_path = os.path.join(gdrive_path, "sentence_scorer_cnn.pt")
torch.save(model_cnn.state_dict(), cnn_model_path)
print(f"\n✅ CNN model saved to: {cnn_model_path}")


In [ ]:
# Cell: Full Evaluation of CNN Sentence Importance Model
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)
import seaborn as sns
import matplotlib.pyplot as plt

print("\n--- Comprehensive Evaluation: CNN Sentence Scorer ---")

# --- Step 1: Inference ---
model_cnn.eval()
y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for inputs, labels in dataloader_cnn:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model_cnn(inputs)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())
        y_prob.extend(probs.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

# --- Step 2: Metrics ---
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
f0_5 = fbeta_score(y_true, y_pred, beta=0.5)
f2 = fbeta_score(y_true, y_pred, beta=2)
auc = roc_auc_score(y_true, y_prob)
avg_precision = average_precision_score(y_true, y_prob)

print(f"\n📊 CNN Model Performance Metrics:")
print(f"Accuracy   : {accuracy:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"Recall     : {recall:.4f}")
print(f"F1 Score   : {f1:.4f}")
print(f"F0.5 Score : {f0_5:.4f}  (Precision-focused)")
print(f"F2 Score   : {f2:.4f}  (Recall-focused)")
print(f"ROC-AUC    : {auc:.4f}")
print(f"PR-AUC     : {avg_precision:.4f}")

# --- Step 3: Classification Report ---
print("\n--- Detailed Classification Report ---")
print(classification_report(y_true, y_pred, target_names=["Not Important (0)", "Important (1)"]))

# --- Step 4: Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion Matrix - CNN Sentence Scorer")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

# --- Step 5: ROC Curve ---
fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - CNN Sentence Scorer")
plt.legend()
plt.show()

# --- Step 6: Precision–Recall Curve ---
precisions, recalls, _ = precision_recall_curve(y_true, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(recalls, precisions, color="purple", label=f"PR Curve (AP = {avg_precision:.2f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curve - CNN Sentence Scorer")
plt.legend()
plt.show()

# --- Step 7: Correlation Heatmap ---
results_df = np.vstack([y_true, y_pred, y_prob]).T
corr = np.corrcoef(results_df.T)
plt.figure(figsize=(5, 4))
sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    xticklabels=["True", "Pred", "Prob"],
    yticklabels=["True", "Pred", "Prob"]
)
plt.title("Correlation Heatmap (True vs Pred vs Prob)")
plt.show()

# --- Step 8: Feature Importance (Filter Activations) ---
model_cnn.eval()
with torch.no_grad():
    sample_batch = X_cnn_tensor.to(device)
    activations = []
    for conv in model_cnn.convs:
        act = torch.relu(conv(sample_batch))
        pooled = torch.max(act, dim=2)[0]
        activations.append(pooled.mean(dim=0).cpu().numpy())

feature_importance = np.mean(np.array(activations), axis=0)
plt.figure(figsize=(10, 4))
plt.bar(range(len(feature_importance)), feature_importance)
plt.title("Feature Importance (CNN Filter Activations)")
plt.xlabel("Filter Index")
plt.ylabel("Avg Activation Strength")
plt.show()


In [ ]:
print(f"\nRNN Avg Loss (Last Epoch): {total_loss/len(dataloader_rnn):.4f}")
print(f"CNN Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")


Initial test

In [ ]:
# Cell: Test the Full Two-Stage DL Pipeline

# --- Part A: Load all necessary models and vocab ---
print("--- Loading all models and data for the final pipeline ---")
import json
import torch.nn.functional as F

# 1. Load the Keyword CNN (architecture must be redefined)
class KeywordCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_filters, filter_sizes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([nn.Conv1d(in_channels=embedding_dim, out_channels=n_filters, kernel_size=fs, padding='same') for fs in filter_sizes])
        self.fc = nn.Linear(n_filters * len(filter_sizes), 1)
        self.dropout = nn.Dropout(0.5)
    def forward(self, text):
        embedded = self.embedding(text).permute(0, 2, 1)
        conved = [F.relu(conv(embedded)) for conv in self.convs]
        cat = self.dropout(torch.cat(conved, dim=1)).permute(0, 2, 1)
        return self.fc(cat).squeeze(-1)

vocab_path = os.path.join(gdrive_path, "keyword_vocab_dialogsum.json")
with open(vocab_path, 'r') as f:
    vocab = json.load(f)
PAD_IDX = vocab['<PAD>']
INPUT_DIM = len(vocab)
keyword_model = KeywordCNN(INPUT_DIM, 100, 50, [1,2,3], PAD_IDX).to(device)
keyword_model.load_state_dict(torch.load(os.path.join(gdrive_path, "keyword_cnn_model_dialogsum.pt")))
keyword_model.eval()
print(" Keyword CNN loaded.")

# 2. Load the Sentence Scoring RNN (architecture must be redefined)
class SentenceScorerRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, n_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, n_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x):
        x = x.unsqueeze(1)
        gru_out, _ = self.gru(x)
        return self.fc(gru_out[:, -1, :]).squeeze(-1)
scorer_model = SentenceScorerRNN(input_dim=embedding_dim).to(device)
scorer_model.load_state_dict(torch.load(os.path.join(gdrive_path, "sentence_scorer_rnn.pt")))
scorer_model.eval()
print(" Sentence Scoring RNN loaded.")


# --- Part B: The Full Pipeline Logic ---
def get_deep_learning_summary(conversation_text, cnn_model, rnn_model, vocab, glove, num_sentences=3):
    sentences = [s.strip() for s in conversation_text.strip().split('\n') if s.strip()]

    # Stage 1: Use CNN to find all keywords in the entire conversation
    all_keywords = set()
    for sentence in sentences:
        words = sentence.lower().split()
        tokenized = [vocab.get(word, vocab['<UNK>']) for word in words]
        input_tensor = torch.LongTensor(tokenized).unsqueeze(0).to(device)
        with torch.no_grad():
            predictions = torch.sigmoid(cnn_model(input_tensor))
        for i, word in enumerate(words):
            if i < len(predictions[0]) and predictions[0, i] > 0.5:
                all_keywords.add(word)

    # Stage 2: Use RNN to score each sentence's importance
    scored_sentences = []
    for sentence in sentences:
        sentence_vec = torch.FloatTensor(sentence_to_vector(sentence)).unsqueeze(0).to(device)
        with torch.no_grad():
            importance_score = rnn_model(sentence_vec).item()

        if ': ' in sentence:
            speaker, dialogue = sentence.split(': ', 1)
            scored_sentences.append({'speaker': speaker, 'dialogue': dialogue, 'score': importance_score})

    # Get the top N sentences based on the RNN's scores
    top_sentences = sorted(scored_sentences, key=lambda x: x['score'], reverse=True)[:num_sentences]

    return top_sentences, all_keywords


# --- Part C: Run on Sample and Display ---
conversation_text = """
Amanda: Hey, did you get the latest project update? The deadline has been moved up to next Friday.
Ben: What? Next Friday? That's impossible, we're still waiting on the final assets from the design team.
Amanda: I know, I flagged that risk. They promised the assets will be delivered by tomorrow morning.
Ben: Okay, that's a tight turnaround. The main blocker is the video rendering, it takes at least 48 hours.
Amanda: I agree. The action item for us is to start the video rendering process the second we get the assets.
"""

summary_points, detected_keywords = get_deep_learning_summary(conversation_text, keyword_model, scorer_model, vocab, glove_embeddings, num_sentences=3)

print("\n" + "="*80)
print("              DEEP LEARNING PIPELINE SUMMARY (CNN + RNN)")
print("="*80)
print(f"\nDetected Keywords (via CNN): {', '.join(sorted(list(detected_keywords)))}")
print("\n--- Key Discussion Points (selected by RNN) ---")

for point in summary_points:
    words_in_dialogue = set(point['dialogue'].lower().split())
    explanation_keywords = detected_keywords.intersection(words_in_dialogue)

    print(f"\n🗣️  **{point['speaker']} said:**")
    print(f"   '{point['dialogue']}'")
    if explanation_keywords:
        print(f"   *-> Explanation: This sentence discusses keywords [{', '.join(explanation_keywords)}]*")

print("\n" + "="*80)

In [ ]:
# Cell: Test the Full Two-Stage DL Pipeline with Various Inputs

# --- Part A: Load all necessary models and vocab (This part is the same) ---
print("--- Loading all models and data for the final pipeline ---")
import json
import torch.nn.functional as F

# (Re-defining the model architectures and loading the saved weights...)
# 1. Load Keyword CNN
class KeywordCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_filters, filter_sizes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([nn.Conv1d(in_channels=embedding_dim, out_channels=n_filters, kernel_size=fs, padding='same') for fs in filter_sizes])
        self.fc = nn.Linear(n_filters * len(filter_sizes), 1)
        self.dropout = nn.Dropout(0.5)
    def forward(self, text):
        embedded = self.embedding(text).permute(0, 2, 1)
        conved = [F.relu(conv(embedded)) for conv in self.convs]
        cat = self.dropout(torch.cat(conved, dim=1)).permute(0, 2, 1)
        return self.fc(cat).squeeze(-1)

vocab_path = os.path.join(gdrive_path, "keyword_vocab_dialogsum.json")
with open(vocab_path, 'r') as f:
    vocab = json.load(f)
PAD_IDX = vocab['<PAD>']
INPUT_DIM = len(vocab)
keyword_model = KeywordCNN(INPUT_DIM, 100, 50, [1,2,3], PAD_IDX).to(device)
keyword_model.load_state_dict(torch.load(os.path.join(gdrive_path, "keyword_cnn_model_dialogsum.pt")))
keyword_model.eval()
print(" Keyword CNN loaded.")

# 2. Load Sentence Scoring RNN
class SentenceScorerRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, n_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, n_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x):
        x = x.unsqueeze(1)
        gru_out, _ = self.gru(x)
        return self.fc(gru_out[:, -1, :]).squeeze(-1)
scorer_model = SentenceScorerRNN(input_dim=100).to(device)
scorer_model.load_state_dict(torch.load(os.path.join(gdrive_path, "sentence_scorer_rnn.pt")))
scorer_model.eval()
print(" Sentence Scoring RNN loaded.")


# --- Part B: The Full Pipeline Logic (This part is the same) ---
def get_deep_learning_summary(conversation_text, cnn_model, rnn_model, vocab, glove, num_sentences=3):
    sentences = [s.strip() for s in conversation_text.strip().split('\n') if s.strip()]
    all_keywords = set()
    for sentence in sentences:
        words = sentence.lower().split()
        tokenized = [vocab.get(word, vocab['<UNK>']) for word in words]
        input_tensor = torch.LongTensor(tokenized).unsqueeze(0).to(device)
        with torch.no_grad():
            predictions = torch.sigmoid(cnn_model(input_tensor))
        for i, word in enumerate(words):
            if i < len(predictions[0]) and predictions[0, i] > 0.5:
                all_keywords.add(word)
    def sentence_to_vector(sentence):
        words = sentence.lower().split()
        zero_vector = np.zeros(100)
        word_vectors = [glove_embeddings.get(word, zero_vector) for word in words]
        return np.mean(word_vectors, axis=0) if len(word_vectors) > 0 else zero_vector
    scored_sentences = []
    for sentence in sentences:
        sentence_vec = torch.FloatTensor(sentence_to_vector(sentence)).unsqueeze(0).to(device)
        with torch.no_grad():
            importance_score = rnn_model(sentence_vec).item()
        if ': ' in sentence:
            speaker, dialogue = sentence.split(': ', 1)
            scored_sentences.append({'speaker': speaker, 'dialogue': dialogue, 'score': importance_score})
    top_sentences = sorted(scored_sentences, key=lambda x: x['score'], reverse=True)[:num_sentences]
    return top_sentences, all_keywords


# --- Part C: Choose a Conversation to Test ---

# --- Conversation 1: Planning a Weekend Trip (Casual & Logistical) ---
# This tests the model's ability to pick up locations, activities, and decisions.
conversation_text = """
Priya: Hey Rohan, any plans for the long weekend?
Rohan: Not yet! I was thinking maybe a quick trip somewhere. Any ideas?
Priya: What about a trek to Nandi Hills? The weather should be perfect.
Rohan: That sounds great, but I heard it gets really crowded. How about exploring Chikmagalur instead? We could visit the coffee plantations.
Priya: Ooh, I like that idea. Let's do Chikmagalur. The main challenge will be booking a good homestay on such short notice.
Rohan: True. I'll start looking for homestays right now. The key action item is to book something by this evening.
Priya: Perfect. I'll check the bus schedules and ticket availability. Let's confirm everything tonight.
"""

# --- Conversation 2: Discussing a Technical Problem (Specific & Domain-focused) ---
# This tests if the model can identify technical jargon as keywords.
# conversation_text = """
# Sam: Hey Chloe, I'm stuck on the authentication bug. The API is returning a 401 error.
# Chloe: A 401 error? Did you check if the JWT token is being sent in the header correctly?
# Sam: Yes, the token is there. I think the issue might be with the token's signature. It might be expiring too quickly.
# Chloe: That's a good lead. The main problem is probably the token validation logic on the server.
# Sam: I agree. The immediate action item is to add more logging to the validation function to see where it's failing.
# Chloe: Right. Let's debug that together after lunch. I'll bring my laptop.
# """

# --- Conversation 3: Making a Dinner Appointment (Simple & Transactional) ---
# This tests the model's ability to extract times, dates, and places.
# conversation_text = """
# Anjali: Hi Karan, are you free for dinner sometime this week?
# Karan: I'd love that! I'm busy on Wednesday, but how about Thursday evening?
# Anjali: Thursday works for me. Any preference on where to go? I was thinking of that new Italian place, "La Piazza".
# Karan: I've heard good things! Let's go there. What time is good for you?
# Anjali: Let's make a reservation for 8:00 PM. That should give us plenty of time.
# Karan: Perfect, 8:00 PM at La Piazza on Thursday it is. I'll book the table right now.
# Anjali: Awesome, see you then!
# """


# --- Run the pipeline and display the results ---
summary_points, detected_keywords = get_deep_learning_summary(conversation_text, keyword_model, scorer_model, vocab, glove_embeddings, num_sentences=3)

print("\n" + "="*80)
print("              DEEP LEARNING PIPELINE SUMMARY (CNN + RNN)")
print("="*80)
print(f"\nDetected Keywords (via CNN): {', '.join(sorted(list(detected_keywords)))}")
print("\n--- Key Discussion Points (selected by RNN) ---")

for point in summary_points:
    words_in_dialogue = set(point['dialogue'].lower().split())
    explanation_keywords = detected_keywords.intersection(words_in_dialogue)

    print(f"\n🗣️  **{point['speaker']} said:**")
    print(f"   '{point['dialogue']}'")
    if explanation_keywords:
        print(f"   *-> Explanation: This sentence discusses keywords [{', '.join(explanation_keywords)}]*")

print("\n" + "="*80)

In [ ]:
# Cell 1: Setup Environment and Configure API (Corrected)
print("--- Installing required libraries ---")
!pip install -q -U google-generativeai
!pip install torch datasets numpy tqdm requests scikit-learn --quiet

from google.colab import drive
import os
import google.generativeai as genai
from google.colab import userdata



gdrive_path = "/content/drive/MyDrive/DL_Course_Project"
print(f"\nProject folder is set to: {gdrive_path}")

# Configure the Gemini API using the key from Colab Secrets
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)

    # --- THIS IS THE CORRECTED LINE ---
    # We are using the current, recommended model name: 'gemini-1.5-flash'
    gemini_model = genai.GenerativeModel('gemini-1.5-flash')

    print(" Gemini API configured successfully with the 'gemini-1.5-flash' model.")
except Exception as e:
    print("⚠️ Gemini API key not found or invalid. The final summary (Stage 3) will be skipped.")
    gemini_model = None

print("\n Setup complete.")

In [ ]:
# Cell: The Definitive, Final Testing Pipeline (All-in-One & CORRECTED)
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import google.generativeai as genai
from google.colab import userdata

# --- Part 1: Configure Gemini API with the STABLE Model Name ---
print("--- Configuring Gemini API ---")
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)

    # --- THIS IS THE GUARANTEED FIX ---
    # We are using the stable, long-term support model 'gemini-1.0-pro'.
    # This model is guaranteed to be available in the v1beta API.
    gemini_model = genai.GenerativeModel('gemini-1.0-pro')

    print(" Gemini API configured successfully with the 'gemini-1.0-pro' model.")
except Exception as e:
    print("⚠️ Gemini API key not found or invalid. Final summary will be skipped.")
    gemini_model = None

# --- Part 2: Load All Your Trained Models and Data ---
print("\n--- Loading all your trained models and data ---")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load Keyword CNN
class KeywordCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_filters, filter_sizes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([nn.Conv1d(in_channels=embedding_dim, out_channels=n_filters, kernel_size=fs, padding='same') for fs in filter_sizes])
        self.fc = nn.Linear(n_filters * len(filter_sizes), 1)
        self.dropout = nn.Dropout(0.5)
    def forward(self, text):
        embedded = self.embedding(text).permute(0, 2, 1)
        conved = [F.relu(conv(embedded)) for conv in self.convs]
        cat = self.dropout(torch.cat(conved, dim=1)).permute(0, 2, 1)
        return self.fc(cat).squeeze(-1)
vocab_path = os.path.join(gdrive_path, "keyword_vocab_dialogsum.json")
with open(vocab_path, 'r') as f:
    vocab = json.load(f)
PAD_IDX = vocab['<PAD>']
INPUT_DIM = len(vocab)
keyword_model = KeywordCNN(INPUT_DIM, 100, 50, [1,2,3], PAD_IDX).to(device)
keyword_model.load_state_dict(torch.load(os.path.join(gdrive_path, "keyword_cnn_model_dialogsum.pt")))
keyword_model.eval()
print(" Keyword CNN loaded.")

# Load Sentence Scoring RNN
class SentenceScorerRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, n_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, n_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x):
        x = x.unsqueeze(1)
        gru_out, _ = self.gru(x)
        return self.fc(gru_out[:, -1, :]).squeeze(-1)
scorer_model = SentenceScorerRNN(input_dim=100).to(device)
scorer_model.load_state_dict(torch.load(os.path.join(gdrive_path, "sentence_scorer_rnn.pt")))
scorer_model.eval()
print(" Sentence Scoring RNN loaded.")

# Load GloVe Embeddings
if 'glove_embeddings' not in locals():
    print("Loading GloVe embeddings...")
    import requests, zipfile, io
    glove_zip_url = "http://nlp.stanford.edu/data/glove.6B.zip"
    r = requests.get(glove_zip_url)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall("/content/glove")
    glove_file_path = "/content/glove/glove.6B.100d.txt"
    glove_embeddings = {}
    with open(glove_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_embeddings[word] = vector
    print(" GloVe embeddings loaded.")

# --- Part 3: Define Input Conversation ---
conversation_text = """
Amanda: Hey, did you get the latest project update? The deadline has been moved up to next Friday.
Ben: What? Next Friday? That's impossible, we're still waiting on the final assets from the design team.
Amanda: I know, I flagged that risk. They promised the assets will be delivered by tomorrow morning.
Ben: Okay, that's a tight turnaround. The main blocker is the video rendering, it takes at least 48 hours.
Amanda: I agree. The action item for us is to start the video rendering process the second we get the assets.
"""

# --- Part 4: Define and Run the Full Pipeline ---
def get_final_summary_pipeline(conversation_text, num_sentences=3):
    # Stage 1 & 2
    sentences = [s.strip() for s in conversation_text.strip().split('\n') if s.strip()]
    all_keywords = set()
    for sentence in sentences:
        words = sentence.lower().split()
        tokenized = [vocab.get(word, vocab['<UNK>']) for word in words]
        input_tensor = torch.LongTensor(tokenized).unsqueeze(0).to(device)
        with torch.no_grad():
            predictions = torch.sigmoid(keyword_model(input_tensor))
        for i, word in enumerate(words):
            if i < len(predictions[0]) and predictions[0, i] > 0.5:
                all_keywords.add(word)
    def sentence_to_vector(sentence):
        words = sentence.lower().split()
        zero_vector = np.zeros(100)
        word_vectors = [glove_embeddings.get(word, zero_vector) for word in words]
        return np.mean(word_vectors, axis=0) if len(word_vectors) > 0 else zero_vector
    scored_sentences = []
    for sentence in sentences:
        sentence_vec = torch.FloatTensor(sentence_to_vector(sentence)).unsqueeze(0).to(device)
        with torch.no_grad():
            importance_score = scorer_model(sentence_vec).item()
        if ': ' in sentence:
            speaker, dialogue = sentence.split(': ', 1)
            scored_sentences.append({'speaker': speaker, 'dialogue': dialogue, 'score': importance_score, 'full_sentence': sentence})
    top_points = sorted(scored_sentences, key=lambda x: x['score'], reverse=True)[:num_sentences]

    # Stage 3
    final_summary_text = "API key not configured or error occurred."
    if gemini_model:
        prompt = f"Based ONLY on the following key points, write a concise, professional summary paragraph of 2-3 sentences:\n\n- {top_points[0]['full_sentence']}\n- {top_points[1]['full_sentence']}\n- {top_points[2]['full_sentence']}"
        try:
            response = gemini_model.generate_content(prompt)
            final_summary_text = response.text
        except Exception as e:
            final_summary_text = f"Error generating summary: {e}"
            print(f"GEMINI API ERROR: {e}")
    return top_points, all_keywords, final_summary_text

# --- Execute the pipeline ---
summary_points, detected_keywords, final_summary = get_final_summary_pipeline(conversation_text)

# --- Part 5: Display ALL Results ---
print("\n" + "="*80)
print("              COMPLETE DEEP LEARNING PIPELINE RESULTS")
print("="*80)
print("\n✨ **Final Summary (Generated by Gemini LLM):**")
print(f"   {final_summary.strip()}")
print("-" * 80)
print("\n🔍 **Explainable AI Details:**")
print(f"\nDetected Keywords (via CNN): {', '.join(sorted(list(detected_keywords)))}")
print("\n--- Key Discussion Points (selected by RNN) ---")
for point in summary_points:
    words_in_dialogue = set(point['dialogue'].lower().split())
    explanation_keywords = detected_keywords.intersection(words_in_dialogue)
    print(f"\n🗣️  **{point['speaker']} said:**")
    print(f"   '{point['dialogue']}'")
    if explanation_keywords:
        print(f"   *-> Why it was chosen: Discusses keywords [{', '.join(explanation_keywords)}]*")
print("\n" + "="*80)

#Feature 2 Done

In [ ]:
# Cell: The Definitive, Final Pipeline (SELF-CONTAINED T5 SUMMARIZER)
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import requests
from transformers import T5Tokenizer, T5ForConditionalGeneration # NEW IMPORTS

# --- Part 1: Load All Your Trained Models and Data (Same as before) ---
print("--- Loading all your trained models and data ---")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load Keyword CNN
class KeywordCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_filters, filter_sizes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.convs = nn.ModuleList([nn.Conv1d(in_channels=embedding_dim, out_channels=n_filters, kernel_size=fs, padding='same') for fs in filter_sizes])
        self.fc = nn.Linear(n_filters * len(filter_sizes), 1)
        self.dropout = nn.Dropout(0.5)
    def forward(self, text):
        embedded = self.embedding(text).permute(0, 2, 1)
        conved = [F.relu(conv(embedded)) for conv in self.convs]
        cat = self.dropout(torch.cat(conved, dim=1)).permute(0, 2, 1)
        return self.fc(cat).squeeze(-1)
vocab_path = os.path.join(gdrive_path, "keyword_vocab_dialogsum.json")
with open(vocab_path, 'r') as f:
    vocab = json.load(f)
PAD_IDX = vocab['<PAD>']
INPUT_DIM = len(vocab)
keyword_model = KeywordCNN(INPUT_DIM, 100, 50, [1,2,3], PAD_IDX).to(device)
keyword_model.load_state_dict(torch.load(os.path.join(gdrive_path, "keyword_cnn_model_dialogsum.pt")))
keyword_model.eval()
print(" Keyword CNN loaded.")

# Load Sentence Scoring RNN
class SentenceScorerRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, n_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, n_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x):
        x = x.unsqueeze(1)
        gru_out, _ = self.gru(x)
        return self.fc(gru_out[:, -1, :]).squeeze(-1)
scorer_model = SentenceScorerRNN(input_dim=100).to(device)
scorer_model.load_state_dict(torch.load(os.path.join(gdrive_path, "sentence_scorer_rnn.pt")))
scorer_model.eval()
print(" Sentence Scoring RNN loaded.")

# Load GloVe Embeddings
if 'glove_embeddings' not in locals():
    print("Loading GloVe embeddings...")
    import zipfile, io
    glove_zip_url = "http://nlp.stanford.edu/data/glove.6B.zip"
    r_glove = requests.get(glove_zip_url)
    z = zipfile.ZipFile(io.BytesIO(r_glove.content))
    z.extractall("/content/glove")
    glove_file_path = "/content/glove/glove.6B.100d.txt"
    glove_embeddings = {}
    with open(glove_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            glove_embeddings[word] = vector
    print(" GloVe embeddings loaded.")

# --- Part 2: NEW - Load the Self-Contained T5 Summarizer ---
print("\n--- Loading the T5 Summarization model ---")
# We use 't5-small', a powerful but lightweight version of Google's T5 model.
t5_model_name = "t5-small"
t5_tokenizer = T5Tokenizer.from_pretrained(t5_model_name)
t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_name).to(device)
print(" T5 model loaded successfully.")


# --- Part 3: Define Input Conversation ---
conversation_text = """
Amanda: Hey, did you get the latest project update? The deadline has been moved up to next Friday.
Ben: What? Next Friday? That's impossible, we're still waiting on the final assets from the design team.
Amanda: I know, I flagged that risk. They promised the assets will be delivered by tomorrow morning.
Ben: Okay, that's a tight turnaround. The main blocker is the video rendering, it takes at least 48 hours.
Amanda: I agree. The action item for us is to start the video rendering process the second we get the assets.
"""

# --- Part 4: Define and Run the Full Pipeline ---
def get_final_summary_pipeline(conversation_text, num_sentences=3):
    # Stage 1 & 2 (Your trained models)
    sentences = [s.strip() for s in conversation_text.strip().split('\n') if s.strip()]
    all_keywords = set()
    for sentence in sentences:
        words = sentence.lower().split()
        tokenized = [vocab.get(word, vocab['<UNK>']) for word in words]
        input_tensor = torch.LongTensor(tokenized).unsqueeze(0).to(device)
        with torch.no_grad():
            predictions = torch.sigmoid(keyword_model(input_tensor))
        for i, word in enumerate(words):
            if i < len(predictions[0]) and predictions[0, i] > 0.5:
                all_keywords.add(word)
    def sentence_to_vector(sentence):
        words = sentence.lower().split()
        zero_vector = np.zeros(100)
        word_vectors = [glove_embeddings.get(word, zero_vector) for word in words]
        return np.mean(word_vectors, axis=0) if len(word_vectors) > 0 else zero_vector
    scored_sentences = []
    for sentence in sentences:
        sentence_vec = torch.FloatTensor(sentence_to_vector(sentence)).unsqueeze(0).to(device)
        with torch.no_grad():
            importance_score = scorer_model(sentence_vec).item()
        if ': ' in sentence:
            speaker, dialogue = sentence.split(': ', 1)
            scored_sentences.append({'speaker': speaker, 'dialogue': dialogue, 'score': importance_score, 'full_sentence': sentence})
    top_points = sorted(scored_sentences, key=lambda x: x['score'], reverse=True)[:num_sentences]

    # Stage 3 (Self-Contained T5 Summarizer)
    # 1. Combine the key points into a single block of text for the model.
    key_points_text = " ".join([point['full_sentence'] for point in top_points])
    # 2. T5 requires a prefix to tell it what task to perform.
    prompt = f"summarize: {key_points_text}"
    # 3. Tokenize the prompt and move it to the GPU.
    inputs = t5_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)
    # 4. Generate the summary.
    with torch.no_grad():
        summary_ids = t5_model.generate(inputs.input_ids, max_length=100, min_length=20, length_penalty=2.0, num_beams=4, early_stopping=True)
    # 5. Decode the generated tokens back into a text string.
    final_summary_text = t5_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    return top_points, all_keywords, final_summary_text

# --- Execute the pipeline ---
summary_points, detected_keywords, final_summary = get_final_summary_pipeline(conversation_text)

# --- Part 5: Display ALL Results ---
print("\n" + "="*80)
print("              COMPLETE DEEP LEARNING PIPELINE RESULTS")
print("="*80)
print("\n✨ **Final Summary (Generated by T5 Transformer):**")
print(f"   {final_summary.strip()}")
print("-" * 80)
print("\n🔍 **Explainable AI Details:**")
print(f"\nDetected Keywords (via CNN): {', '.join(sorted(list(detected_keywords)))}")
print("\n--- Key Discussion Points (selected by RNN) ---")
for point in summary_points:
    words_in_dialogue = set(point['dialogue'].lower().split())
    explanation_keywords = detected_keywords.intersection(words_in_dialogue)
    print(f"\n🗣️  **{point['speaker']} said:**")
    print(f"   '{point['dialogue']}'")
    if explanation_keywords:
        print(f"   *-> Why it was chosen: Discusses keywords [{', '.join(explanation_keywords)}]*")
print("\n" + "="*80)

##Feature 3 Trial

In [ ]:
# Cell 1: Setup Environment
print("--- Installing required libraries ---")
!pip install torch librosa numpy tqdm requests scikit-learn --quiet

from google.colab import drive
import os

print("\n--- Mounting Google Drive ---")
drive.mount('/content/drive')

gdrive_path = "/content/drive/MyDrive/DL_Course_Project"
os.makedirs(gdrive_path, exist_ok=True)
print(f"\n Setup complete. Project folder is set to: {gdrive_path}")

In [ ]:
# Cell 1: Setup Environment (Updated with resampy)
print("--- Installing required libraries ---")
# Added 'resampy' which is needed by librosa for resampling
!pip install torch librosa numpy tqdm requests scikit-learn resampy --quiet

from google.colab import drive
import os

print("\n--- Mounting Google Drive ---")
drive.mount('/content/drive')

gdrive_path = "/content/drive/MyDrive/DL_Course_Project"
os.makedirs(gdrive_path, exist_ok=True)
print(f"\n Setup complete. Project folder is set to: {gdrive_path}")

In [ ]:
# Cell 2: (DEFINITIVE & SELF-CONTAINED with TORCHAUDIO) Feature 3
import requests
import zipfile
import io
import os
import glob
import numpy as np
from tqdm.auto import tqdm
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
# --- NEW IMPORTS ---
import torchaudio
import torchaudio.transforms as T
# --- REMOVED librosa ---

print("--- Step 2: Downloading, Verifying, and Extracting Features using TORCHAUDIO ---")

# --- Part A: Download and Extract (Same as before) ---
dataset_zip_url = "https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"
extract_path = "/content/ravdess_data"
expected_data_path = extract_path

try:
    print("Downloading dataset...")
    r = requests.get(dataset_zip_url, stream=True)
    r.raise_for_status()
    print("Extracting dataset...")
    z = zipfile.ZipFile(io.BytesIO(r.content))
    z.extractall(extract_path)
    print(f"Dataset extracted to '{extract_path}'")

    # --- Part B: IMMEDIATE Verification (Same as before) ---
    print("\nVerifying extraction...")
    wav_files = glob.glob(os.path.join(expected_data_path, "**", "*.wav"), recursive=True)
    num_files = len(wav_files)
    if num_files < 1000:
        raise FileNotFoundError(f"Dataset verification failed, only found {num_files} WAV files.")
    else:
        print(f" Verification successful! Found {num_files} .wav files.")

    # --- Part C: Feature Extraction (Using TORCHAUDIO) ---
    print("\nStarting feature extraction using torchaudio...")
    emotion_map = {
        "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
        "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised"
    }
    features = []
    labels = []
    processed_files = 0
    failed_files = 0
    target_sample_rate = 16000
    n_mfcc = 40

    # --- Initialize torchaudio transforms ---
    # This replaces librosa.feature.mfcc
    mfcc_transform = T.MFCC(
        sample_rate=target_sample_rate,
        n_mfcc=n_mfcc,
        melkwargs={'n_fft': 400, 'hop_length': 160, 'n_mels': 64} # Common MFCC parameters
    )
    resampler = None # Will be initialized if needed

    for file_path in tqdm(wav_files, desc="Processing Audio Files"):
        try:
            file_name = os.path.basename(file_path)
            emotion_code = file_name.split('-')[2]
            emotion = emotion_map[emotion_code]

            # --- Load audio using torchaudio ---
            # waveform is a tensor, info contains metadata like sample_rate
            waveform, sample_rate = torchaudio.load(file_path)

            # --- Resample using torchaudio if necessary ---
            if sample_rate != target_sample_rate:
                if resampler is None or resampler.orig_freq != sample_rate:
                     # Initialize resampler only when needed for a specific rate
                     resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
                waveform = resampler(waveform)

            # Ensure waveform is mono
            if waveform.shape[0] > 1:
                waveform = torch.mean(waveform, dim=0, keepdim=True)

            # Limit duration (optional, but helps consistency)
            max_len_samples = int(3 * target_sample_rate) # 3 seconds
            if waveform.shape[1] > max_len_samples:
                 waveform = waveform[:, :max_len_samples]

            # --- Extract MFCCs using torchaudio ---
            mfccs_tensor = mfcc_transform(waveform) # Shape: [channel, n_mfcc, time]

            # --- Calculate mean and convert to numpy ---
            # Average across the time dimension (dim=2)
            mfccs = torch.mean(mfccs_tensor.squeeze(0), dim=1).numpy()

            if mfccs.size > 0:
                features.append(mfccs)
                labels.append(emotion)
                processed_files += 1
            else:
                failed_files += 1
        except Exception as e:
            if failed_files < 5:
                print(f"\nERROR processing file {file_name}: {e}")
            failed_files += 1
            continue

    if processed_files == 0:
         print("\n CRITICAL ERROR: No audio files were successfully processed.")
    else:
        print(f"\n Feature extraction complete!")
        print(f"   Successfully processed: {processed_files} audio files.")
        print(f"   Skipped or failed: {failed_files} files.")


    # --- Part D: Prepare Data for PyTorch Model (Same as before) ---
    print("\n--- Part D: Preparing data for PyTorch model ---")
    X = np.array(features)
    y_text = np.array(labels)
    le = LabelEncoder()
    y = le.fit_transform(y_text)
    encoder_save_path = os.path.join(gdrive_path, "emotion_label_encoder.npy")
    np.save(encoder_save_path, le.classes_)
    print(f"Label encoder saved.")
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
    test_dataset = TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test))
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    print(f"Data ready. Classes: {le.classes_}")

    # --- Part E: Define and Train the Emotion Classifier (Same as before) ---
    class EmotionClassifier(nn.Module):
        def __init__(self, num_features, num_classes):
            super().__init__()
            self.model = nn.Sequential(
                nn.Linear(num_features, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
                nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.5),
                nn.Linear(128, num_classes)
            )
        def forward(self, x):
            return self.model(x)

    num_features = X_train.shape[1]
    num_classes = len(le.classes_)
    model = EmotionClassifier(num_features=num_features, num_classes=num_classes)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    print(f"\n--- Part E: Starting training on {device} ---")
    num_epochs = 75
    best_accuracy = 0.0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for batch_features, batch_labels in train_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_features.size(0)

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_features, batch_labels in test_loader:
                batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
                outputs = model(batch_features)
                _, predicted = torch.max(outputs.data, 1)
                total += batch_labels.size(0)
                correct += (predicted == batch_labels).sum().item()
        accuracy = 100 * correct / total
        print(f"Epoch {epoch + 1}/{num_epochs} | Train Loss: {train_loss/len(train_loader.dataset):.4f} | Test Accuracy: {accuracy:.2f}%")

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            model_save_path = os.path.join(gdrive_path, "emotion_classifier_mlp_best.pt")
            torch.save(model.state_dict(), model_save_path)
            # print(f"   ---> New best model saved (Accuracy: {best_accuracy:.2f}%)")

    print(f"\n Training complete! Best test accuracy: {best_accuracy:.2f}%")
    print(f"\nFinal model saved successfully to: '{model_save_path}'")

except FileNotFoundError as e:
    print(f"\n ERROR during setup/verification: {e}")
except Exception as e:
    print(f" An unexpected error occurred: {e}")

Testing

In [ ]:
# Cell 1: Setup Environment
print("--- Installing required libraries ---")
!pip install torch torchaudio numpy scikit-learn requests --quiet

from google.colab import drive
import os

print("\n--- Mounting Google Drive ---")
drive.mount('/content/drive')

gdrive_path = "/content/drive/MyDrive/DL_Course_Project"
print(f"\n Setup complete. Project folder is set to: {gdrive_path}")

In [ ]:
# Cell 2: Load Trained Model and Label Encoder
import torch
import torch.nn as nn
import numpy as np

print("--- Step 2: Loading the trained model and label encoder ---")

# --- Part A: Load the Label Encoder ---
encoder_path = os.path.join(gdrive_path, "emotion_label_encoder.npy")
try:
    emotion_classes = np.load(encoder_path, allow_pickle=True)
    print(f" Label encoder loaded. Classes: {emotion_classes}")
except FileNotFoundError:
    print(f" ERROR: Label encoder file not found at '{encoder_path}'. Please ensure it was saved correctly during training.")
    # Stop execution if encoder is missing
    raise

# --- Part B: Load Your Trained Model ---
# 1. Define the model architecture EXACTLY as during training
class EmotionClassifier(nn.Module):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.model(x)

# 2. Instantiate the model
num_features = 40 # We used 40 MFCCs during training
num_classes = len(emotion_classes)
inference_model = EmotionClassifier(num_features=num_features, num_classes=num_classes)

# 3. Load the saved weights from your Google Drive
model_path = os.path.join(gdrive_path, "emotion_classifier_mlp_best.pt")
try:
    inference_model.load_state_dict(torch.load(model_path))
except FileNotFoundError:
    print(f" ERROR: Model weights file not found at '{model_path}'. Please ensure it was saved correctly during training.")
    raise
except RuntimeError as e:
    print(f" ERROR: Could not load model weights. Ensure the model architecture here matches the one used for training.")
    print(f"   Details: {e}")
    raise


# 4. Set the model to evaluation mode and move to device
device = "cuda" if torch.cuda.is_available() else "cpu"
inference_model.to(device)
inference_model.eval()

print(f"\n Trained model loaded successfully and is ready for inference on {device}!")

In [ ]:
# Cell 3, Option A: Upload a test audio file
from google.colab import files
import torchaudio

print("Please upload a short WAV or MP3 audio file to test.")
uploaded = files.upload()

# Get the name of the uploaded file
test_audio_path = list(uploaded.keys())[0]

print(f"\nFile '{test_audio_path}' uploaded successfully.")

In [ ]:
# Cell 4: Process Audio and Predict Emotion
import torchaudio
import torchaudio.transforms as T

# Check if a valid audio path exists from the previous step
if 'test_audio_path' in locals() and test_audio_path and os.path.exists(test_audio_path):
    print("--- Step 4: Processing audio and predicting emotion ---")

    # --- Part A: Process the Audio File ---
    target_sample_rate = 16000
    n_mfcc = 40

    # Initialize transforms
    mfcc_transform = T.MFCC(
        sample_rate=target_sample_rate,
        n_mfcc=n_mfcc,
        melkwargs={'n_fft': 400, 'hop_length': 160, 'n_mels': 64}
    )
    resampler = None

    try:
        waveform, sample_rate = torchaudio.load(test_audio_path)

        if sample_rate != target_sample_rate:
            if resampler is None or resampler.orig_freq != sample_rate:
                 resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
            waveform = resampler(waveform)

        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Extract MFCCs and calculate the mean
        mfccs_tensor = mfcc_transform(waveform)
        mfccs_mean = torch.mean(mfccs_tensor.squeeze(0), dim=1) # Average across time

        # Ensure the feature vector has the correct dimension (40)
        if mfccs_mean.shape[0] != num_features:
             raise ValueError(f"Extracted MFCCs have incorrect shape: {mfccs_mean.shape}. Expected ({num_features},)")

        # Prepare for model (add batch dimension and move to device)
        input_features = mfccs_mean.unsqueeze(0).to(device)

        # --- Part B: Predict with the Model ---
        with torch.no_grad():
            outputs = inference_model(input_features)
            # Get the index of the highest probability
            _, predicted_idx = torch.max(outputs.data, 1)
            predicted_emotion = emotion_classes[predicted_idx.item()]

        # --- Display the result ---
        print("\n" + "="*50)
        print("              PREDICTION RESULTS")
        print("="*50)
        print(f"\nPREDICTED EMOTION: {predicted_emotion.upper()}")
        # If using Option B, show the expected emotion
        if 'sample_url' in locals():
             print(f"(Expected Emotion: Happy)")
        print("="*50)

    except FileNotFoundError:
        print(f" ERROR: Audio file not found at '{test_audio_path}'.")
    except ValueError as ve:
        print(f" ERROR during feature extraction: {ve}")
        print("   This might happen if the audio file is too short or corrupted.")
    except Exception as e:
        print(f" An unexpected error occurred during processing or prediction: {e}")
else:
     print(" ERROR: No valid 'test_audio_path' found. Please run one of the cells in Step 3 first.")

In [ ]:
# Cell 4: Process Audio, Predict Emotion, and MAP TO VIBE
import torchaudio
import torchaudio.transforms as T
import torch # Ensure torch is imported

# Check if a valid audio path exists from the previous step
if 'test_audio_path' in locals() and test_audio_path and os.path.exists(test_audio_path):
    print("--- Step 4: Processing audio, predicting emotion, and mapping to vibe ---")

    # --- Part A: Process the Audio File (Same as before) ---
    target_sample_rate = 16000
    n_mfcc = 40
    mfcc_transform = T.MFCC(
        sample_rate=target_sample_rate,
        n_mfcc=n_mfcc,
        melkwargs={'n_fft': 400, 'hop_length': 160, 'n_mels': 64}
    )
    resampler = None

    try:
        waveform, sample_rate = torchaudio.load(test_audio_path)
        if sample_rate != target_sample_rate:
            if resampler is None or resampler.orig_freq != sample_rate:
                 resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
            waveform = resampler(waveform)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        mfccs_tensor = mfcc_transform(waveform)
        mfccs_mean = torch.mean(mfccs_tensor.squeeze(0), dim=1)
        if mfccs_mean.shape[0] != num_features:
             raise ValueError(f"Extracted MFCCs have incorrect shape: {mfccs_mean.shape}. Expected ({num_features},)")
        input_features = mfccs_mean.unsqueeze(0).to(device)

        # --- Part B: Predict Basic Emotion with the Model (Same as before) ---
        with torch.no_grad():
            outputs = inference_model(input_features)
            probabilities = torch.softmax(outputs, dim=1) # Get probabilities for each class
            _, predicted_idx = torch.max(outputs.data, 1)
            predicted_emotion = emotion_classes[predicted_idx.item()]
            confidence = probabilities[0, predicted_idx.item()].item() * 100

        # --- Part C: NEW - Map the Predicted Emotion to a Vibe Category ---
        vibe = "Neutral / Calm" # Default
        if predicted_emotion in ["happy", "surprised"]:
            vibe = "Positive / Engaged 😊"
        elif predicted_emotion in ["angry", "disgust"]:
            vibe = "Negative / Frustrated 😠"
        elif predicted_emotion in ["fearful", "sad"]:
            vibe = "Hesitant / Uncertain 🤔"
        # 'neutral' and 'calm' map to the default

        # --- Display the results ---
        print("\n" + "="*60)
        print("              VIBE ANALYSIS RESULTS")
        print("="*60)
        print(f"\nPredicted Basic Emotion: {predicted_emotion.capitalize()} ({confidence:.1f}% confidence)")
        print(f"Inferred Conversational Vibe: {vibe}")
        # If using Option B, show the expected emotion
        if 'sample_url' in locals():
             print(f"(Expected Emotion for sample: Happy)")
        print("="*60)

    except FileNotFoundError:
        print(f" ERROR: Audio file not found at '{test_audio_path}'.")
    except ValueError as ve:
        print(f" ERROR during feature extraction: {ve}")
    except Exception as e:
        print(f" An unexpected error occurred: {e}")
else:
     print(" ERROR: No valid 'test_audio_path' found. Please run one of the cells in Step 3 first.")

#Feature 3 ,More Accurate:

In [ ]:
# Cell 1: Setup Environment
print("--- Installing required libraries ---")
!pip install torch torchaudio numpy tqdm requests scikit-learn --quiet

from google.colab import drive
import os

print("\n--- Mounting Google Drive ---")
drive.mount('/content/drive')

gdrive_path = "/content/drive/MyDrive/DL_Course_Project"
os.makedirs(gdrive_path, exist_ok=True)
print(f"\n Setup complete. Project folder is set to: {gdrive_path}")

In [ ]:
# Cell 2: Download and Verify the RAVDESS Emotion Dataset
import requests
import zipfile
import io
import os
import glob # Using glob for robust file counting

print("--- Step 2: Downloading and Verifying the RAVDESS audio dataset ---")
url = "https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"
extract_path = "/content/ravdess_data"
expected_data_path = extract_path # Actor folders are directly inside

try:
    print("Downloading dataset...")
    r = requests.get(url, stream=True); r.raise_for_status()
    print("Extracting dataset...")
    z = zipfile.ZipFile(io.BytesIO(r.content)); z.extractall(extract_path)
    print(f"Dataset extracted to '{extract_path}'")
    wav_files = glob.glob(os.path.join(expected_data_path, "**", "*.wav"), recursive=True)
    num_files = len(wav_files)
    if num_files < 1000: raise FileNotFoundError(f"Verification Failed! Only found {num_files} WAV files.")
    print(f" Verification successful! Found {num_files} .wav files.")
except Exception as e:
    print(f" ERROR: {e}")
    wav_files = [] # Ensure wav_files exists even on error

In [ ]:
# Cell 3: Extract MFCC SEQUENCES
import torchaudio
import torchaudio.transforms as T
import numpy as np
from tqdm.auto import tqdm

print("--- Step 3: Extracting MFCC sequences (no averaging) ---")

emotion_map = { "01": "neutral", "02": "calm", "03": "happy", "04": "sad", "05": "angry", "06": "fearful", "07": "disgust", "08": "surprised" }
features = [] # List to hold MFCC sequences (tensors)
labels = []
processed_files = 0
failed_files = 0
target_sample_rate = 16000
n_mfcc = 40
resampler = None

mfcc_transform = T.MFCC(
    sample_rate=target_sample_rate, n_mfcc=n_mfcc,
    melkwargs={'n_fft': 400, 'hop_length': 160, 'n_mels': 64}
)

if 'wav_files' in locals() and wav_files:
    for file_path in tqdm(wav_files, desc="Processing Audio Files"):
        try:
            file_name = os.path.basename(file_path)
            emotion_code = file_name.split('-')[2]
            emotion = emotion_map[emotion_code]
            waveform, sample_rate = torchaudio.load(file_path)

            if sample_rate != target_sample_rate:
                if resampler is None or resampler.orig_freq != sample_rate:
                     resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
                waveform = resampler(waveform)

            if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
            max_len_samples = int(3 * target_sample_rate)
            if waveform.shape[1] > max_len_samples: waveform = waveform[:, :max_len_samples]
            if waveform.shape[1] < 400: # Skip very short files for sequence models
                 failed_files +=1; continue

            # --- KEY CHANGE: Keep the sequence ---
            mfccs_tensor = mfcc_transform(waveform) # Shape: [1, n_mfcc, time]
            # Transpose to [time, n_mfcc] for RNN
            mfccs_sequence = mfccs_tensor.squeeze(0).transpose(0, 1)

            if mfccs_sequence.shape[0] > 0: # Check sequence length
                features.append(mfccs_sequence) # Append the tensor directly
                labels.append(emotion)
                processed_files += 1
            else: failed_files += 1
        except Exception as e:
            if failed_files < 5: print(f"\nERROR processing {file_name}: {e}")
            failed_files += 1; continue

    if processed_files > 0:
        print(f"\n Feature extraction complete! Successfully processed {processed_files} sequences.")
        print(f"   Skipped or failed: {failed_files} files.")
    else: print("\n CRITICAL ERROR: No sequences processed.")
else: print("\n ERROR: No .wav files found.")

In [ ]:
# Cell 4: Prepare Sequential Data for Training (with Padding)
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch.nn.utils.rnn import pad_sequence

if processed_files > 0:
    print("--- Step 4: Preparing sequential data for PyTorch RNN model ---")

    # 1. Encode Labels (Same as before)
    y_text = np.array(labels)
    le = LabelEncoder()
    y = le.fit_transform(y_text)
    encoder_save_path = os.path.join(gdrive_path, "emotion_label_encoder_rnn.npy")
    np.save(encoder_save_path, le.classes_)
    print(f"Label encoder saved.")

    # 2. Split indices for train/test (We split before creating the Dataset)
    indices = list(range(len(features)))
    train_indices, test_indices = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)

    # 3. Create a custom PyTorch Dataset
    class EmotionSequenceDataset(Dataset):
        def __init__(self, features, labels, indices):
            self.features = [features[i] for i in indices]
            self.labels = [labels[i] for i in indices]
        def __len__(self):
            return len(self.labels)
        def __getitem__(self, idx):
            return self.features[idx], self.labels[idx]

    train_dataset = EmotionSequenceDataset(features, y, train_indices)
    test_dataset = EmotionSequenceDataset(features, y, test_indices)

    # 4. Define the Custom Collate Function for Padding
    def collate_fn_pad(batch):
        # Separate features and labels
        sequences = [item[0] for item in batch]
        targets = torch.LongTensor([item[1] for item in batch])
        # Get lengths before padding
        lengths = torch.LongTensor([len(seq) for seq in sequences])
        # Pad sequences
        padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0.0)
        return padded_sequences, targets, lengths

    # 5. Create DataLoaders with the custom collate function
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn_pad)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn_pad)

    print("\n Sequential data is ready for training!")
    print(f"Train samples: {len(train_dataset)} | Test samples: {len(test_dataset)}")
else: print("\nSkipping data preparation.")

In [ ]:
# Cell 5: Define and Train the GRU Emotion Classifier
import torch.nn as nn

if processed_files > 0:
    print("--- Step 5: Defining and Training the GRU Classifier ---")

    # --- Define the GRU Model ---
    class EmotionGRU(nn.Module):
        def __init__(self, input_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
            super().__init__()
            self.gru = nn.GRU(input_dim, hidden_dim, n_layers, batch_first=True,
                              bidirectional=bidirectional, dropout=dropout if n_layers > 1 else 0)
            # Multiply hidden_dim by 2 if bidirectional
            self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x, x_lengths):
            # x shape: (batch_size, seq_len, input_dim)
            # Pack sequence -> Process with GRU -> Unpack sequence
            # This handles variable lengths efficiently.
            packed_embedded = nn.utils.rnn.pack_padded_sequence(x, x_lengths.cpu(), batch_first=True, enforce_sorted=False)
            packed_output, hidden = self.gru(packed_embedded)
            # output, output_lengths = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)

            # We can use the final hidden state for classification
            # Concatenate the final forward and backward hidden states
            if self.gru.bidirectional:
                # hidden shape: (n_layers * 2, batch_size, hidden_dim)
                # Get hidden state of last layer, both directions
                hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
            else:
                hidden = self.dropout(hidden[-1,:,:]) # Get hidden state of last layer

            return self.fc(hidden)


    # --- Prepare for Training ---
    INPUT_DIM = n_mfcc
    HIDDEN_DIM = 128
    OUTPUT_DIM = len(le.classes_)
    N_LAYERS = 2
    BIDIRECTIONAL = True
    DROPOUT = 0.5

    model = EmotionGRU(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, BIDIRECTIONAL, DROPOUT)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5) # Slightly adjusted lr

    # --- The Training Loop ---
    print(f"Starting training on {device}...")
    num_epochs = 40 # Might need fewer epochs than MLP
    best_accuracy = 0.0

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        for seqs, targets, lengths in train_loader:
            seqs, targets = seqs.to(device), targets.to(device)
            optimizer.zero_grad()
            predictions = model(seqs, lengths) # Pass lengths to model
            loss = criterion(predictions, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * seqs.size(0)

        # --- Evaluation ---
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for seqs, targets, lengths in test_loader:
                seqs, targets = seqs.to(device), targets.to(device)
                predictions = model(seqs, lengths) # Pass lengths to model
                _, predicted = torch.max(predictions.data, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()

        accuracy = 100 * correct / total
        print(f"Epoch {epoch + 1}/{num_epochs} | Train Loss: {train_loss/len(train_loader.dataset):.4f} | Test Accuracy: {accuracy:.2f}%")

        if accuracy > best_accuracy:
            best_accuracy = accuracy
            model_save_path = os.path.join(gdrive_path, "emotion_classifier_gru_best.pt")
            torch.save(model.state_dict(), model_save_path)
            # print(f"   ---> New best model saved (Accuracy: {best_accuracy:.2f}%)")

    print(f"\n Training complete! Best test accuracy: {best_accuracy:.2f}%")
    print(f"\nFinal GRU model saved successfully to: '{model_save_path}'")
else: print("\nSkipping model training.")

Test

In [ ]:
# Cell 1: Setup Environment
print("--- Installing required libraries ---")
!pip install torch torchaudio numpy tqdm requests scikit-learn --quiet

from google.colab import drive
import os

print("\n--- Mounting Google Drive ---")
drive.mount('/content/drive')

gdrive_path = "/content/drive/MyDrive/DL_Course_Project"
print(f"\n Setup complete. Project folder is set to: {gdrive_path}")

In [ ]:
# Cell 2: Load Trained GRU Model and Label Encoder
import torch
import torch.nn as nn
import numpy as np
import os

print("--- Step 2: Loading the trained GRU model and label encoder ---")

# --- Part A: Load the Label Encoder ---
# Make sure this is the encoder saved during the GRU training
encoder_path = os.path.join(gdrive_path, "emotion_label_encoder_rnn.npy")
try:
    emotion_classes = np.load(encoder_path, allow_pickle=True)
    print(f" Label encoder loaded. Classes: {emotion_classes}")
except FileNotFoundError:
    print(f" ERROR: Label encoder file not found at '{encoder_path}'.")
    raise

# --- Part B: Load Your Trained GRU Model ---
# 1. Define the GRU model architecture EXACTLY as during training
INPUT_DIM = 40 # n_mfcc
HIDDEN_DIM = 128
OUTPUT_DIM = len(emotion_classes)
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5

class EmotionGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, n_layers, batch_first=True,
                          bidirectional=bidirectional, dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, x_lengths):
        packed_embedded = nn.utils.rnn.pack_padded_sequence(x, x_lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.gru(packed_embedded)
        if self.gru.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
        return self.fc(hidden)

# 2. Instantiate the model
inference_model = EmotionGRU(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, BIDIRECTIONAL, DROPOUT)

# 3. Load the saved weights from your Google Drive
# Make sure this is the GRU model file
model_path = os.path.join(gdrive_path, "emotion_classifier_gru_best.pt")
try:
    inference_model.load_state_dict(torch.load(model_path))
except FileNotFoundError:
    print(f" ERROR: Model weights file not found at '{model_path}'.")
    raise
except RuntimeError as e:
    print(f" ERROR: Could not load model weights. Architecture mismatch?")
    print(f"   Details: {e}")
    raise

# 4. Set the model to evaluation mode and move to device
device = "cuda" if torch.cuda.is_available() else "cpu"
inference_model.to(device)
inference_model.eval()

print(f"\n Trained GRU model loaded successfully on {device}!")

In [ ]:
# Cell 3, Option A: Upload a test audio file
from google.colab import files
import torchaudio

print("Please upload a short WAV or MP3 audio file to test.")
uploaded = files.upload()
test_audio_path = list(uploaded.keys())[0]
print(f"\nFile '{test_audio_path}' uploaded.")

In [ ]:
# Cell 4: Process Audio SEQUENCE and Predict Emotion/Vibe with GRU
import torchaudio
import torchaudio.transforms as T

if 'test_audio_path' in locals() and test_audio_path and os.path.exists(test_audio_path):
    print("--- Step 4: Processing audio sequence and predicting vibe with GRU ---")

    # --- Part A: Process the Audio File into a SEQUENCE ---
    target_sample_rate = 16000
    n_mfcc = 40
    mfcc_transform = T.MFCC(
        sample_rate=target_sample_rate, n_mfcc=n_mfcc,
        melkwargs={'n_fft': 400, 'hop_length': 160, 'n_mels': 64}
    )
    resampler = None

    try:
        waveform, sample_rate = torchaudio.load(test_audio_path)
        if sample_rate != target_sample_rate:
            if resampler is None or resampler.orig_freq != sample_rate:
                 resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
            waveform = resampler(waveform)
        if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Extract MFCC sequence
        mfccs_tensor = mfcc_transform(waveform) # Shape: [1, n_mfcc, time]
        # Transpose to [time, n_mfcc]
        mfccs_sequence = mfccs_tensor.squeeze(0).transpose(0, 1)

        # Prepare input for the GRU model
        input_sequence = mfccs_sequence.unsqueeze(0).to(device) # Add batch dim: [1, time, n_mfcc]
        input_length = torch.LongTensor([mfccs_sequence.shape[0]]) # Get the actual length

        # --- Part B: Predict with the GRU Model ---
        with torch.no_grad():
            outputs = inference_model(input_sequence, input_length)
            probabilities = torch.softmax(outputs, dim=1)
            _, predicted_idx = torch.max(outputs.data, 1)
            predicted_emotion = emotion_classes[predicted_idx.item()]
            confidence = probabilities[0, predicted_idx.item()].item() * 100

        # --- Part C: Map to Vibe (Same logic as before) ---
        vibe = "Neutral / Calm"
        if predicted_emotion in ["happy", "surprised"]: vibe = "Positive / Engaged 😊"
        elif predicted_emotion in ["angry", "disgust"]: vibe = "Negative / Frustrated 😠"
        elif predicted_emotion in ["fearful", "sad"]: vibe = "Hesitant / Uncertain 🤔"

        # --- Display the result ---
        print("\n" + "="*60)
        print("              GRU VIBE ANALYSIS RESULTS")
        print("="*60)
        print(f"\nPredicted Basic Emotion: {predicted_emotion.capitalize()} ({confidence:.1f}% confidence)")
        print(f"Inferred Conversational Vibe: {vibe}")
        if 'sample_url' in locals(): print(f"(Expected Emotion for sample: Happy)")
        print("="*60)

    except FileNotFoundError: print(f" ERROR: Audio file not found: '{test_audio_path}'.")
    except Exception as e: print(f" An unexpected error occurred: {e}")
else: print(" ERROR: No valid 'test_audio_path'. Run Step 3 first.")

Getting a detailed vibe

In [ ]:
# Cell 1a: Ensure librosa and resampy are installed
!pip install librosa resampy --quiet
print(" Librosa and resampy installed/updated.")

In [ ]:
# Cell 1: Setup Environment
print("--- Installing required libraries ---")
!pip install torch torchaudio numpy tqdm requests scikit-learn --quiet

from google.colab import drive
import os

print("\n--- Mounting Google Drive ---")
drive.mount('/content/drive')

gdrive_path = "/content/drive/MyDrive/DL_Course_Project"
print(f"\n Setup complete. Project folder is set to: {gdrive_path}")

In [ ]:
# Cell 2: Load Trained GRU Model and Label Encoder
import torch
import torch.nn as nn
import numpy as np
import os

print("--- Step 2: Loading the trained GRU model and label encoder ---")

# --- Part A: Load the Label Encoder ---
# Make sure this is the encoder saved during the GRU training
encoder_path = os.path.join(gdrive_path, "emotion_label_encoder_rnn.npy")
try:
    emotion_classes = np.load(encoder_path, allow_pickle=True)
    print(f" Label encoder loaded. Classes: {emotion_classes}")
except FileNotFoundError:
    print(f" ERROR: Label encoder file not found at '{encoder_path}'.")
    raise

# --- Part B: Load Your Trained GRU Model ---
# 1. Define the GRU model architecture EXACTLY as during training
INPUT_DIM = 40 # n_mfcc
HIDDEN_DIM = 128
OUTPUT_DIM = len(emotion_classes)
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5

class EmotionGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, n_layers, batch_first=True,
                          bidirectional=bidirectional, dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, x_lengths):
        packed_embedded = nn.utils.rnn.pack_padded_sequence(x, x_lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.gru(packed_embedded)
        if self.gru.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
        return self.fc(hidden)

# 2. Instantiate the model
inference_model = EmotionGRU(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, BIDIRECTIONAL, DROPOUT)

# 3. Load the saved weights from your Google Drive
# Make sure this is the GRU model file
model_path = os.path.join(gdrive_path, "emotion_classifier_gru_best.pt")
try:
    inference_model.load_state_dict(torch.load(model_path))
except FileNotFoundError:
    print(f" ERROR: Model weights file not found at '{model_path}'.")
    raise
except RuntimeError as e:
    print(f" ERROR: Could not load model weights. Architecture mismatch?")
    print(f"   Details: {e}")
    raise

# 4. Set the model to evaluation mode and move to device
device = "cuda" if torch.cuda.is_available() else "cpu"
inference_model.to(device)
inference_model.eval()

print(f"\n Trained GRU model loaded successfully on {device}!")

In [ ]:
# Cell 3, Option A: Upload a test audio file
from google.colab import files
import torchaudio

print("Please upload a short WAV or MP3 audio file to test.")
uploaded = files.upload()
test_audio_path = list(uploaded.keys())[0]
print(f"\nFile '{test_audio_path}' uploaded.")

In [ ]:
# Cell 4: Process Audio, Extract Features, Predict, Map to DESCRIPTIVE VIBE
import torchaudio
import torchaudio.transforms as T
import torch
import librosa # Need librosa for pitch and energy
import numpy as np

if 'test_audio_path' in locals() and test_audio_path and os.path.exists(test_audio_path):
    print("--- Step 4: Processing audio, extracting features, predicting, and mapping to DESCRIPTIVE vibe ---")

    # --- Part A: Process Audio & Extract Features ---
    target_sample_rate = 16000
    n_mfcc = 40
    mfcc_transform = T.MFCC(
        sample_rate=target_sample_rate, n_mfcc=n_mfcc,
        melkwargs={'n_fft': 400, 'hop_length': 160, 'n_mels': 64}
    )
    resampler = None

    try:
        # Load with Librosa as well for acoustic features (requires numpy array)
        audio_np, sr_lib = librosa.load(test_audio_path, sr=target_sample_rate) # Load directly at target rate
        waveform = torch.tensor(audio_np).unsqueeze(0) # Convert to torch tensor for MFCC/model
        sample_rate = target_sample_rate # We loaded at the target rate

        # --- Extract Acoustic Features (Pitch Var, Energy) ---
        # Pitch (F0) and Voiced Flag
        f0, voiced_flag, voiced_probs = librosa.pyin(audio_np, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
        # Get only voiced F0 values and calculate standard deviation
        voiced_f0 = f0[voiced_flag]
        pitch_variation = np.std(voiced_f0) if len(voiced_f0) > 1 else 0.0

        # Energy (RMS)
        rms_energy = np.mean(librosa.feature.rms(y=audio_np)[0])

        # --- Extract MFCC Sequence (using torchaudio) ---
        if waveform.shape[0] > 1: waveform = torch.mean(waveform, dim=0, keepdim=True)
        mfccs_tensor = mfcc_transform(waveform)
        mfccs_sequence = mfccs_tensor.squeeze(0).transpose(0, 1)
        input_sequence = mfccs_sequence.unsqueeze(0).to(device)
        input_length = torch.LongTensor([mfccs_sequence.shape[0]])

        # --- Part B: Predict Emotion Probabilities with GRU Model ---
        with torch.no_grad():
            outputs = inference_model(input_sequence, input_length)
            probabilities = torch.softmax(outputs, dim=1).squeeze().cpu().numpy()
            predicted_idx = np.argmax(probabilities)
            predicted_emotion = emotion_classes[predicted_idx]
            confidence = probabilities[predicted_idx] * 100

        # --- Part C: NEW - Descriptive Vibe Mapping ---

        def map_to_descriptive_vibe(probs, classes, pitch_std, energy):
            """ Maps emotion probabilities and acoustic features to a descriptive vibe. """
            top_idx = np.argmax(probs)
            top_prob = probs[top_idx]
            top_emotion = classes[top_idx]

            # Define thresholds (these might need tuning)
            HIGH_CONF = 0.70
            MOD_CONF = 0.50
            HIGH_PITCH_VAR = 50 # Hz standard deviation
            LOW_PITCH_VAR = 20
            HIGH_ENERGY = 0.05 # RMS energy level
            LOW_ENERGY = 0.01

            # Rule-based inference
            if top_emotion in ["happy", "surprised"] and top_prob > HIGH_CONF and pitch_std > HIGH_PITCH_VAR and energy > HIGH_ENERGY:
                return f"Excited / Enthusiastic 🎉 (High {top_emotion.capitalize()}, expressive pitch, high energy)"
            elif top_emotion in ["happy", "surprised", "calm"] and top_prob > MOD_CONF and energy > LOW_ENERGY:
                 return f"Positive / Engaged 🙂 (Detected {top_emotion.capitalize()}, moderate energy)"
            elif top_emotion in ["angry", "disgust"] and top_prob > HIGH_CONF and energy > HIGH_ENERGY:
                return f"Frustrated / Agitated 😠 (High {top_emotion.capitalize()}, high energy)"
            elif top_emotion in ["neutral"] and top_prob > HIGH_CONF and energy > HIGH_ENERGY and pitch_std > LOW_PITCH_VAR:
                 return f"Confident / Assertive 😎 (Strong neutral, high energy, moderate pitch var)"
            elif top_emotion in ["neutral", "calm"] and top_prob > HIGH_CONF and pitch_std < LOW_PITCH_VAR:
                return f"Informational / Neutral 😐 (High {top_emotion.capitalize()}, monotonous pitch)"
            elif top_emotion in ["sad", "fearful"] and top_prob > HIGH_CONF and energy < LOW_ENERGY:
                return f"Concerned / Sad 😟 (High {top_emotion.capitalize()}, low energy)"
            elif top_emotion in ["fearful", "sad"] and top_prob > MOD_CONF : # Removed energy constraint
                 return f"Hesitant / Uncertain 🤔 (Detected {top_emotion.capitalize()})"
            elif top_emotion == "calm" and top_prob > HIGH_CONF and energy < LOW_ENERGY:
                 return f"Calm / Relaxed 😌 (High Calm, low energy)"
            # Fallback if confidence is low or mixed signals
            elif top_prob < MOD_CONF :
                 return f"Uncertain / Mixed Vibe (Low confidence: {top_emotion.capitalize()}?)"
            else: # Default if none of the specific rules match
                return f"Neutral / Calm (Detected {top_emotion.capitalize()})"


        vibe_description = map_to_descriptive_vibe(probabilities, emotion_classes, pitch_variation, rms_energy)

        # --- Display the result ---
        print("\n" + "="*80)
        print("                     DESCRIPTIVE VIBE ANALYSIS RESULTS (GRU + Acoustics)")
        print("="*80)
        print(f"\nPredicted Top Emotion: {predicted_emotion.capitalize()} ({confidence:.1f}% confidence)")
        print(f"\nAcoustic Cues: Pitch Variation={pitch_variation:.2f} Hz | RMS Energy={rms_energy:.4f}")
        print(f"\nInferred Conversational Vibe: {vibe_description}")
        # Optional: Display top 3 probabilities
        top3_indices = np.argsort(probabilities)[::-1][:3]
        top3_probs = probabilities[top3_indices]
        top3_emotions = emotion_classes[top3_indices]
        print("\nTop 3 Emotion Probabilities:")
        for i in range(3): print(f"  - {top3_emotions[i].capitalize()}: {top3_probs[i]*100:.1f}%")
        if 'sample_url' in locals(): print(f"\n(Expected Emotion for sample: Happy)")
        print("="*80)

    except FileNotFoundError: print(f" ERROR: Audio file not found: '{test_audio_path}'.")
    except Exception as e: print(f" An unexpected error occurred: {e}")
else: print(" ERROR: No valid 'test_audio_path'. Run Step 3 first.")